# KYC — RAW OCR Pipeline · Qwen3.6-27B-FP8 · H100 · Domino Data Lab

**Pipeline version : `KYC_RAW_OCR_V1_0_QWEN3_6_27B_FP8`**

---

## Objective

> **Can Qwen3.6-27B-FP8 reliably read the complete content of poor-quality KYC scanned pages?**

This notebook answers that single question and nothing else.

### What this notebook IS

```text
1 PDF page  ->  1 conservative render  ->  1 Qwen call  ->  1 literal transcription
```

* **RAW OCR only.** Literal, multilingual (FR / AR / EN), verbatim transcription.
* **Measurable.** Every stage of every page is timed independently so that a slow
  run can be attributed to a *specific* cause instead of a single opaque total.
* **Safe by default.** Diagnostic mode processes **one** reproducible customer and
  enforces a hard cap on the number of model calls.
* **Resumable.** Page-level checkpoints keyed by SHA-256 + pipeline version.

### What this notebook is NOT

It contains **no** page classification, **no** structured field extraction, **no**
`CHAMPS_ATTENDUS`, **no** salary logic, **no** domiciliation / pre-domiciliation
references, **no** Excel business matching, **no** name matching, **no** date
tolerance, **no** permit cropping rules, **no** database comparison.

```text
STAGE 1  (this notebook)  = document reading / RAW OCR
STAGE 2  (out of scope)   = structured information extraction
```

Separating the two is deliberate: you cannot debug an extraction schema while the
underlying transcription quality is still unknown.

---

## Relationship to `dom.ipynb`

`dom.ipynb` already proved that this exact checkpoint loads and performs
image-text inference in this exact Domino environment. Its **technical
foundations are reused unchanged**; only its business logic is dropped.

| Reused verbatim from `dom.ipynb` | Why |
|---|---|
| `AutoProcessor` + `AutoModelForImageTextToText` | Proven compatible with the local checkpoint. No speculative model class. |
| `FineGrainedFP8Config(dequantize=True)` + `dtype=torch.bfloat16` | Proven working FP8 load path. Not re-quantized, not "modernized". |
| `device_map="auto"`, `trust_remote_code=True`, `low_cpu_mem_usage=True` | Same loading contract. |
| `processor.tokenizer.padding_side = "left"` | Correct for decoder-only generation. |
| `torch.backends.cuda.matmul.allow_tf32 = True` | Same TF32 policy. |
| `apply_chat_template(..., enable_thinking=False)` + `TypeError` fallback | Same thinking-disabled contract. |
| `messages = [{"role": "user", "content": [{"type": "image", ...}, {"type": "text", ...}]}]` | Same multimodal message construction. |
| `do_sample=False`, `repetition_penalty=1.0`, `pad_token_id=eos` | Same deterministic generation. |
| Input-token trimming `out[0][input_len:]` before decode | Same correct decode contract. |
| PyMuPDF (`fitz`), dynamic `doc.page_count`, RGB `Image.frombytes`, aspect-preserving resize | Same rendering stack. |
| `white_ratio()` / `is_blank()` | Same blank-page heuristic. |
| SHA-256 checkpoints, resumability, per-file error isolation, `gc.collect()` + `empty_cache()` | Same robustness patterns. |
| Two-tier render (standard / high-definition) | Same vision-token control philosophy. |
| Package **detection** instead of blind reinstall | Same "do not break Domino CUDA" policy. |

### Deliberate, justified deviations

Each of these is argued in the Markdown cell that precedes the code.

1. **Per-page rendering instead of whole-PDF materialisation** — RAW OCR is
   page-checkpointed, so holding every page of every PDF in RAM buys nothing and
   costs memory; it also makes per-page render timing exact. (§8)
2. **Input tensors moved to the model's *resolved* input-embedding device**
   instead of a hard-coded `"cuda"` — identical behaviour on a single H100,
   but correct and loudly diagnosed if `device_map="auto"` ever offloads. (§11, §13)
3. **One call per page, no classification call** — the logical document type is
   already known from the filename, so the classify-then-extract architecture of
   `dom.ipynb` would double the call count for no benefit. (§15, §37)
4. **Generic conservative preprocessing** replaces the document-specific crops,
   which were tied to the previous business case. (§9)
5. **A repetition stopping criterion** is added so the `!!!!!!!!` failure mode
   costs ~3 s instead of running to `max_new_tokens`. (§13, §14)

---

## Notebook map

| § | Content |
|---|---|
| 1 | Environment / package validation (detect, never blindly reinstall) |
| 2 | Imports |
| 3 | Configuration (`dataclass`) |
| 4 | Environment and H100 diagnostics |
| 5 | Safe ZIP extraction (path-traversal hardened) |
| 6 | KYC inventory (controlled filename matching, duplicates, missing) |
| 7 | Inventory reports (CSV + JSON) |
| 8 | PDF / image utilities |
| 9 | Generic conservative preprocessing |
| 10 | Model loading (FP8) |
| 11 | Model / inference diagnostics |
| 12 | RAW OCR prompt |
| 13 | Qwen multimodal input + single call |
| 14 | Output-degeneration detector |
| 15 | Single-page RAW OCR |
| 16 | Single-page benchmark (cold vs warm + projections) |
| 17 | Single-PDF processor |
| 18 | Single-customer processor |
| 19 | Checkpoint / resume utilities |
| 20 | Diagnostic customer selection and call plan |
| 21 | Diagnostic run |
| 22 | RAW OCR inspection |
| 23 | Performance analysis |
| 24 | Output writers |
| 25 | Full-dataset runner (disabled by default) |
| 26 | Optional experimental batching (disabled) + final summary |

**Run the cells in order.** Sections 1–20 define and inspect; section 21 is the
first cell that performs a real multi-page run.

## 1 — Environment / package validation

Same philosophy as `dom.ipynb`: **detect, report, and fail loudly — never blindly
reinstall**. Re-installing `torch`, `transformers`, `accelerate` or a CUDA runtime
inside a Domino workspace is the fastest way to break a working GPU image.

The cell below only *reads* installed versions. If something is missing it raises
with an explicit list, and you install **only** what is listed, manually.

In [ ]:
# -----------------------------------------------------------------------------
# Package detection ONLY. Nothing is installed here.
#
# If a package is reported ABSENT, install it manually and deliberately, e.g.:
#     %pip install -q -U 'transformers>=4.57.0'
# Never reinstall torch / CUDA inside Domino.
# -----------------------------------------------------------------------------
import sys
from importlib import metadata

REQUIRED_PACKAGES = {
    "torch": "2.0",
    "transformers": "4.57",
    "accelerate": "0.30",
    "PyMuPDF": "1.23",
    "Pillow": "9.0",
    "pandas": "1.5",
    "numpy": "1.23",
}

OPTIONAL_PACKAGES = {
    "psutil": "5.9",  # host RAM reporting only
}


def _version_tuple(text):
    """Best-effort numeric version tuple, tolerant of suffixes such as 4.57.0.dev0."""
    parts = []
    for chunk in str(text).split("."):
        digits = ""
        for char in chunk:
            if char.isdigit():
                digits += char
            else:
                break
        if digits == "":
            break
        parts.append(int(digits))
    return tuple(parts) if parts else (0,)


print("Python :", sys.version.replace("\n", " "))
print("\nRequired packages:")

missing = []
too_old = []
for package_name, minimum in REQUIRED_PACKAGES.items():
    try:
        found = metadata.version(package_name)
    except metadata.PackageNotFoundError:
        missing.append(package_name)
        print(f"  {package_name:15s} ABSENT        | minimum advised {minimum}")
        continue

    flag = "ok"
    if _version_tuple(found) < _version_tuple(minimum):
        flag = "BELOW MINIMUM"
        too_old.append(f"{package_name} {found} < {minimum}")
    print(f"  {package_name:15s} {found:14s}| minimum advised {minimum:8s} | {flag}")

print("\nOptional packages:")
for package_name, minimum in OPTIONAL_PACKAGES.items():
    try:
        found = metadata.version(package_name)
        print(f"  {package_name:15s} {found:14s}| minimum advised {minimum}")
    except metadata.PackageNotFoundError:
        print(f"  {package_name:15s} ABSENT        | optional, pipeline still runs")

if missing:
    raise RuntimeError(
        "Missing packages: "
        + ", ".join(missing)
        + ". Install ONLY these packages in the Domino environment, then re-run this cell."
    )

if too_old:
    print("\n[WARNING] Versions below the advised minimum:")
    for item in too_old:
        print("   -", item)
    print("  The pipeline will still attempt to run, but FP8 / Qwen3.6 support")
    print("  requires transformers >= 4.57. Upgrade transformers only if needed.")

print("\nPackage check complete. Environment left untouched.")

## 2 — Imports

`fitz` is the PyMuPDF module name in this Domino image (as in `dom.ipynb`); it is
not changed. `openpyxl` is **not** imported — there is no Excel output in a RAW
OCR baseline.

In [ ]:
import gc
import hashlib
import json
import logging
import math
import os
import platform
import random
import re
import shutil
import statistics
import sys
import time
import unicodedata
import zipfile
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass, field
from datetime import datetime
from importlib import metadata
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import fitz  # PyMuPDF — same import as dom.ipynb
import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageFilter
from transformers import AutoModelForImageTextToText, AutoProcessor
from transformers import StoppingCriteria, StoppingCriteriaList

try:
    import psutil
except ImportError:  # optional
    psutil = None

# Pillow >= 9.1 moved the resampling enums; support both without breaking.
try:
    RESAMPLE_LANCZOS = Image.Resampling.LANCZOS
    RESAMPLE_BICUBIC = Image.Resampling.BICUBIC
except AttributeError:  # Pillow < 9.1
    RESAMPLE_LANCZOS = Image.LANCZOS
    RESAMPLE_BICUBIC = Image.BICUBIC

print("Imports OK")
print("Python       :", sys.version.split()[0])
print("PyMuPDF      :", getattr(fitz, "__doc__", "loaded").splitlines()[0] if getattr(fitz, "__doc__", None) else "loaded")
print("Torch        :", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU          :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

## 3 — Configuration

Everything tunable lives in one `dataclass`. No magic numbers are buried in the
pipeline code.

### Why these rendering defaults

`dom.ipynb` used `PDF_ZOOM = 3.0` / `IMAGE_MAX_SIZE = 1400` for **typed** contracts
and escalated to `4.5` / `2200` for the degraded bilingual permit sheet. KYC scans
are systematically harder (ID cards photographed inside an A4 page, MRZ lines,
handwriting), so the standard pass is raised slightly:

| | zoom | max side | A4 result | ≈ vision tokens | ≈ effective DPI |
|---|---|---|---|---|---|
| Standard | 3.0 | **1600** | 1131 × 1600 | ≈ 2 300 | ≈ 135 |
| HD fallback | 4.5 | **2200** | 1556 × 2200 | ≈ 4 400 | ≈ 185 |

Vision tokens are consumed in **prefill**, which is compute-bound and cheap on an
H100 (a few hundred milliseconds). What actually costs wall-clock time is
**decode**, which is memory-bandwidth-bound. That asymmetry drives the next choice.

### Why `MAX_NEW_TOKENS_RAW_OCR = 2048`

A 27 B model dequantised to bf16 holds ≈ 54 GB of weights. Decoding is bandwidth
bound, so on an H100 (≈ 3.3 TB/s HBM) the ceiling is ≈ 60 tok/s and the realistic
HF-Transformers figure is **≈ 25–40 tok/s**. Therefore:

```text
max_new_tokens = 2048  ->  worst case ≈ 50–80 s/page
max_new_tokens = 6000  ->  worst case ≈ 150–240 s/page  ->  ≈ 2–3 h for 50 pages
```

**This is the single most likely root cause of previous multi-hour OCR runs**: a
degenerate generation (`!!!!!!!!`) that never emits EOS runs to the cap, and the
cap was too high. 2048 tokens is ≈ 7 000–8 000 characters of French — more than a
dense A4 page of legal text — so legitimate pages finish on EOS well below it.
Truncation is *measured* (`stop_reason == "MAX_TOKENS"`), not assumed: if real
pages are being cut off, raise the value with evidence, not by default.

The degenerate case is additionally bounded by a repetition stopping criterion
(§13), which halts a `!!!!!!!!` loop after ~96 tokens instead of 2048.

### Why `MAX_PIXELS = 3600 * 32 * 32`

The processor ceiling must cover the **HD** render (1556 × 2200 ≈ 3.42 Mpx),
otherwise the fallback would be silently downscaled and would carry no extra
information. `MIN_PIXELS` is kept identical to `dom.ipynb`.

In [ ]:
PIPELINE_VERSION = "KYC_RAW_OCR_V1_0_QWEN3_6_27B_FP8"

# The five logical KYC documents. The historical business spelling
# "CARTON SIGNATUTE.PDF" is preserved verbatim and is the canonical key.
TARGET_DOCUMENTS = [
    "JUSTIFICATIF IDENTITE.PDF",
    "JUSTIFICATIF DOMICILE.PDF",
    "CONVENTION COMPTE.PDF",
    "FATCA.PDF",
    "CARTON SIGNATUTE.PDF",
]

# Controlled aliases -> canonical logical type.
# Exact match AFTER normalisation only. No fuzzy matching, no edit distance:
# an unrelated PDF must never be mapped onto one of the five categories.
DOCUMENT_ALIASES = {
    "CARTON SIGNATURE.PDF": "CARTON SIGNATUTE.PDF",   # corrected spelling
    "JUSTIFICATIF D IDENTITE.PDF": "JUSTIFICATIF IDENTITE.PDF",
    "JUSTIFICATIF DE DOMICILE.PDF": "JUSTIFICATIF DOMICILE.PDF",
    "CONVENTION DE COMPTE.PDF": "CONVENTION COMPTE.PDF",
}


@dataclass
class PipelineConfig:
    """Single source of truth for the KYC RAW OCR pipeline."""

    # ---------------------------------------------------------------- model --
    MODEL_PATH: str = "/domino/edv/modelhub/ModelHub-model-huggingface-Qwen/Qwen3.6-27B-FP8/main"

    # ------------------------------------------------------------------ i/o --
    ZIP_PATH: Path = Path("/mnt/data/kyc_documents.zip")
    ZIP_SEARCH_DIRS: List[Path] = field(default_factory=lambda: [
        Path("/mnt/data"),
        Path("/mnt/data/kyc"),
        Path("/domino/datasets/local"),
        Path.cwd(),
        Path.cwd() / "data",
    ])
    EXTRACT_DIR: Path = Path("/mnt/data/kyc_extracted")
    OUTPUT_DIR: Path = Path("/mnt/data/kyc_raw_ocr_out")

    # ------------------------------------------------------------ safety net --
    DIAGNOSTIC_MODE: bool = True
    RANDOM_SEED: int = 42
    NUM_DIAGNOSTIC_CUSTOMERS: int = 1
    MAX_QWEN_CALLS: int = 20
    DIAGNOSTIC_PREFER_COMPLETE: bool = True   # prefer a customer with more target docs
    RUN_FULL_DATASET: bool = False            # hard gate for section 25
    PRINT_TRANSCRIPTION: bool = True          # print RAW OCR in diagnostic mode
    PRINT_TRANSCRIPTION_MAX_CHARS: int = 4000

    # --------------------------------------------------------------- render --
    RENDER_ZOOM_STANDARD: float = 3.0
    IMAGE_MAX_SIZE_STANDARD: int = 1600
    RENDER_ZOOM_HD: float = 4.5
    IMAGE_MAX_SIZE_HD: int = 2200

    # ------------------------------------------------------------ processor --
    MIN_PIXELS: int = 4 * 32 * 32
    MAX_PIXELS: int = 3600 * 32 * 32          # must cover the HD render

    # ----------------------------------------------------------- generation --
    MAX_NEW_TOKENS_RAW_OCR: int = 2048
    MAX_NEW_TOKENS_RAW_OCR_HD: int = 3072
    REPETITION_PENALTY: float = 1.0           # unchanged from dom.ipynb
    ENABLE_REPETITION_STOP: bool = True
    REPETITION_STOP_WINDOW: int = 96          # tokens inspected
    REPETITION_STOP_UNIQUE: int = 2           # <= N distinct ids in the window -> stop

    # -------------------------------------------------------- preprocessing --
    ENABLE_PREPROCESSING: bool = True
    ENABLE_AUTO_CROP: bool = True
    ENABLE_DESKEW: bool = True
    ENABLE_CONTRAST: bool = True
    ENABLE_SHARPEN: bool = True
    ENABLE_DENOISE: bool = False              # off: destroys thin strokes / diacritics
    ENABLE_ORIENTATION: bool = False          # off by default, see section 9

    AUTO_CROP_INK_THRESHOLD: int = 200        # pixel < this = ink
    AUTO_CROP_MARGIN_FRACTION: float = 0.012  # safety pad kept around the ink box
    AUTO_CROP_MIN_GAIN: float = 0.08          # crop only if it removes >= 8% of area
    AUTO_CROP_MIN_KEPT_FRACTION: float = 0.25 # never keep less than 25% of the page
    DESKEW_MAX_ANGLE: float = 3.0             # degrees explored, +/-
    DESKEW_STEP: float = 0.25
    DESKEW_MIN_ANGLE_APPLY: float = 0.40      # below this, rotating is pure loss
    CONTRAST_RANGE_TRIGGER: int = 200         # stretch only if p2..p98 span < this
    CONTRAST_PERCENTILE_LOW: float = 2.0
    CONTRAST_PERCENTILE_HIGH: float = 98.0
    SHARPEN_RADIUS: float = 1.2
    SHARPEN_PERCENT: int = 60
    SHARPEN_THRESHOLD: int = 3
    PREPROCESS_MAX_INK_LOSS: float = 0.25     # revert if preprocessing eats 25% of ink
    ORIENTATION_EVIDENCE_RATIO: float = 3.0   # projection evidence required for a 90 deg fix

    # ------------------------------------------------------------ blank page --
    SKIP_BLANK_PAGES: bool = True
    BLANK_THRESHOLD: float = 0.995            # same heuristic as dom.ipynb

    # -------------------------------------------------------------- fallback --
    ENABLE_HD_FALLBACK: bool = True
    FALLBACK_MIN_INK_RATIO: float = 0.02      # page clearly has content
    FALLBACK_MIN_CHARS: int = 60              # ... but transcription is tiny

    # ------------------------------------------------- degeneration detector --
    DEGEN_MIN_LENGTH: int = 40                # below this, ratio tests are meaningless
    DEGEN_MIN_UNIQUE_CHARS: int = 6
    DEGEN_ENTROPY_MIN: float = 2.0            # bits/char
    DEGEN_MAX_CHAR_RUN: int = 30
    DEGEN_PUNCT_RATIO_MAX: float = 0.55
    DEGEN_PERIOD_MAX: int = 32                # cyclic pattern length probed
    DEGEN_PERIOD_MIN_REPEATS: int = 8
    DEGEN_MAX_LINE_REPEAT: int = 12

    # ------------------------------------------------------------ duplicates --
    PROCESS_DUPLICATES: bool = True           # RAW diagnostic: keep every physical file

    # ------------------------------------------------------------ monitoring --
    SLOW_PAGE_WARNING_S: float = 90.0
    LOW_THROUGHPUT_WARNING_TPS: float = 8.0

    # ------------------------------------------------------------ derived ----
    @property
    def INVENTORY_DIR(self) -> Path:
        return self.OUTPUT_DIR / "inventory"

    @property
    def RAW_OCR_DIR(self) -> Path:
        return self.OUTPUT_DIR / "raw_ocr"

    @property
    def PERFORMANCE_DIR(self) -> Path:
        return self.OUTPUT_DIR / "performance"

    @property
    def CHECKPOINT_DIR(self) -> Path:
        return self.OUTPUT_DIR / "checkpoints"

    @property
    def LOG_DIR(self) -> Path:
        return self.OUTPUT_DIR / "logs"

    @property
    def LOG_PATH(self) -> Path:
        return self.LOG_DIR / "kyc_raw_ocr.log"

    def ensure_dirs(self) -> None:
        for directory in (
            self.OUTPUT_DIR,
            self.INVENTORY_DIR,
            self.RAW_OCR_DIR,
            self.PERFORMANCE_DIR,
            self.CHECKPOINT_DIR,
            self.LOG_DIR,
            self.EXTRACT_DIR,
        ):
            directory.mkdir(parents=True, exist_ok=True)


CFG = PipelineConfig()
CFG.ensure_dirs()

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Session state, initialised here so that sections 22-27 stay runnable whichever
# run path is taken (diagnostic in section 21, or full dataset in section 25).
RUN_RECORDS: List[Dict[str, Any]] = []
RUN_ERRORS: List[Dict[str, Any]] = []
PDF_FAILURES: List[Dict[str, Any]] = []
CUSTOMER_RESULTS: List[Dict[str, Any]] = []
RUN_ELAPSED_S: float = 0.0

random.seed(CFG.RANDOM_SEED)
np.random.seed(CFG.RANDOM_SEED)
torch.manual_seed(CFG.RANDOM_SEED)

print(f"Pipeline version : {PIPELINE_VERSION}")
print(f"Device           : {DEVICE}")
print(f"Model path       : {CFG.MODEL_PATH}")
print(f"ZIP (configured) : {CFG.ZIP_PATH}")
print(f"Extract dir      : {CFG.EXTRACT_DIR}")
print(f"Output dir       : {CFG.OUTPUT_DIR}")
print(f"Diagnostic mode  : {CFG.DIAGNOSTIC_MODE}  (customers={CFG.NUM_DIAGNOSTIC_CUSTOMERS}, max Qwen calls={CFG.MAX_QWEN_CALLS})")
print(f"Full dataset run : {CFG.RUN_FULL_DATASET}")
print(f"Render standard  : zoom={CFG.RENDER_ZOOM_STANDARD} max_side={CFG.IMAGE_MAX_SIZE_STANDARD}")
print(f"Render HD        : zoom={CFG.RENDER_ZOOM_HD} max_side={CFG.IMAGE_MAX_SIZE_HD} (fallback={CFG.ENABLE_HD_FALLBACK})")
print(f"max_new_tokens   : {CFG.MAX_NEW_TOKENS_RAW_OCR} (HD fallback {CFG.MAX_NEW_TOKENS_RAW_OCR_HD})")
print(f"Target documents : {len(TARGET_DOCUMENTS)}")
for _name in TARGET_DOCUMENTS:
    print(f"   - {_name}")

In [ ]:
# -----------------------------------------------------------------------------
# Logging: one file handler + one stream handler, installed exactly once so that
# re-running this cell does not duplicate every log line.
# -----------------------------------------------------------------------------
LOGGER = logging.getLogger("kyc_raw_ocr")
LOGGER.setLevel(logging.INFO)
LOGGER.propagate = False

for _handler in list(LOGGER.handlers):
    LOGGER.removeHandler(_handler)
    try:
        _handler.close()
    except Exception:
        pass

_formatter = logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s", "%Y-%m-%d %H:%M:%S")

_file_handler = logging.FileHandler(CFG.LOG_PATH, encoding="utf-8")
_file_handler.setFormatter(_formatter)
LOGGER.addHandler(_file_handler)

_stream_handler = logging.StreamHandler(stream=sys.stdout)
_stream_handler.setFormatter(logging.Formatter("%(message)s"))
_stream_handler.setLevel(logging.INFO)
LOGGER.addHandler(_stream_handler)


def log(message: str, level: int = logging.INFO) -> None:
    """Single logging entry point used across the pipeline."""
    LOGGER.log(level, message)


log(f"Logging initialised -> {CFG.LOG_PATH}")
log(f"{PIPELINE_VERSION} | session start {datetime.now().isoformat(timespec='seconds')}")

## 4 — Environment and H100 diagnostics

Baseline snapshot taken **before** the model is loaded, so that the VRAM figures
printed after loading (§11) are attributable to the model and nothing else.

In [ ]:
def _safe_version(package_name: str) -> str:
    try:
        return metadata.version(package_name)
    except Exception:
        return "unknown"


def describe_environment() -> Dict[str, Any]:
    """Collect and print the full environment / GPU snapshot."""
    info: Dict[str, Any] = {
        "python_version": sys.version.split()[0],
        "platform": platform.platform(),
        "torch_version": torch.__version__,
        "transformers_version": _safe_version("transformers"),
        "accelerate_version": _safe_version("accelerate"),
        "pymupdf_version": _safe_version("PyMuPDF"),
        "pillow_version": _safe_version("Pillow"),
        "numpy_version": np.__version__,
        "pandas_version": pd.__version__,
        "cuda_available": bool(torch.cuda.is_available()),
        "cuda_runtime_version": torch.version.cuda,
        "cudnn_version": None,
        "gpu_name": None,
        "gpu_compute_capability": None,
        "gpu_total_memory_gb": None,
        "gpu_count": 0,
        "bf16_supported": False,
        "initial_allocated_mb": 0.0,
        "initial_reserved_mb": 0.0,
        "host_ram_total_gb": None,
        "host_ram_available_gb": None,
    }

    try:
        info["cudnn_version"] = torch.backends.cudnn.version()
    except Exception:
        info["cudnn_version"] = None

    if psutil is not None:
        virtual_memory = psutil.virtual_memory()
        info["host_ram_total_gb"] = round(virtual_memory.total / 1e9, 1)
        info["host_ram_available_gb"] = round(virtual_memory.available / 1e9, 1)

    print("=" * 78)
    print("ENVIRONMENT")
    print("=" * 78)
    print(f"Python version        : {info['python_version']}")
    print(f"Platform              : {info['platform']}")
    print(f"PyTorch version       : {info['torch_version']}")
    print(f"Transformers version  : {info['transformers_version']}")
    print(f"Accelerate version    : {info['accelerate_version']}")
    print(f"PyMuPDF version       : {info['pymupdf_version']}")
    print(f"Pillow version        : {info['pillow_version']}")
    print(f"NumPy / pandas        : {info['numpy_version']} / {info['pandas_version']}")
    if info["host_ram_total_gb"] is not None:
        print(f"Host RAM total/free   : {info['host_ram_total_gb']} GB / {info['host_ram_available_gb']} GB")
    else:
        print("Host RAM total/free   : psutil not installed")

    print("-" * 78)
    print(f"CUDA available        : {info['cuda_available']}")
    print(f"CUDA runtime (torch)  : {info['cuda_runtime_version']}")
    print(f"cuDNN version         : {info['cudnn_version']}")

    if not torch.cuda.is_available():
        print("-" * 78)
        print("[FATAL] No CUDA device visible. This pipeline requires a GPU.")
        print("=" * 78)
        return info

    info["gpu_count"] = torch.cuda.device_count()
    info["gpu_name"] = torch.cuda.get_device_name(0)
    major, minor = torch.cuda.get_device_capability(0)
    info["gpu_compute_capability"] = f"{major}.{minor}"
    info["gpu_total_memory_gb"] = round(
        torch.cuda.get_device_properties(0).total_memory / 1e9, 2
    )
    try:
        info["bf16_supported"] = bool(torch.cuda.is_bf16_supported())
    except Exception:
        info["bf16_supported"] = major >= 8

    info["initial_allocated_mb"] = round(torch.cuda.memory_allocated() / 1e6, 1)
    info["initial_reserved_mb"] = round(torch.cuda.memory_reserved() / 1e6, 1)

    print(f"GPU count             : {info['gpu_count']}")
    print(f"GPU name              : {info['gpu_name']}")
    print(f"Compute capability    : {info['gpu_compute_capability']}")
    print(f"Total GPU memory      : {info['gpu_total_memory_gb']} GB")
    print(f"BF16 supported        : {info['bf16_supported']}")
    print(f"Initial allocated VRAM: {info['initial_allocated_mb']} MB")
    print(f"Initial reserved VRAM : {info['initial_reserved_mb']} MB")
    print("=" * 78)

    if not info["bf16_supported"]:
        print("[WARNING] BF16 is not reported as supported. The FP8->bf16 dequantised")
        print("          load path of dom.ipynb assumes bf16 compute.")
    if info["gpu_total_memory_gb"] is not None and info["gpu_total_memory_gb"] < 70:
        print("[WARNING] A 27B checkpoint dequantised to bf16 needs roughly 54 GB of")
        print("          weights plus activations and KV cache. On a GPU below ~70 GB")
        print("          device_map='auto' will offload to CPU and inference will")
        print("          become dramatically slower. See section 11.")

    return info


ENV_INFO = describe_environment()

if not ENV_INFO["cuda_available"]:
    raise RuntimeError("This pipeline requires a CUDA GPU (same constraint as dom.ipynb).")

# Same TF32 policy as dom.ipynb.
torch.backends.cuda.matmul.allow_tf32 = True
print(f"\ntorch.backends.cuda.matmul.allow_tf32 = {torch.backends.cuda.matmul.allow_tf32}")

## 5 — Safe ZIP extraction

`kyc_documents.zip` is untrusted input. A ZIP archive can contain absolute paths
(`/etc/passwd`), parent traversal (`../../`), or symlinks pointing outside the
destination — the classic *Zip Slip*. Every member is therefore resolved against
the destination directory and rejected if it escapes it.

Extraction is **idempotent**: a member already present with the same size is not
rewritten, so re-running the cell after a kernel restart costs nothing.

In [ ]:
def resolve_zip_path(cfg: PipelineConfig = None) -> Path:
    """
    Locate kyc_documents.zip.

    The configured path wins. If it is absent, a short, explicit list of candidate
    directories is searched. Nothing is guessed outside that list.
    """
    cfg = cfg or CFG
    configured = Path(cfg.ZIP_PATH)
    if configured.is_file():
        return configured

    filename = configured.name
    tried = [str(configured)]
    for directory in cfg.ZIP_SEARCH_DIRS:
        candidate = Path(directory) / filename
        tried.append(str(candidate))
        if candidate.is_file():
            log(f"ZIP found outside the configured path: {candidate}")
            return candidate

    raise FileNotFoundError(
        "kyc_documents.zip not found.\nTried:\n  - " + "\n  - ".join(tried)
        + "\nUpload the archive to one of these locations, or set CFG.ZIP_PATH."
    )


def _is_within_directory(directory: Path, target: Path) -> bool:
    """True if `target` resolves inside `directory` (Zip Slip guard)."""
    try:
        directory_resolved = directory.resolve()
        target_resolved = target.resolve()
    except Exception:
        return False
    return str(target_resolved) == str(directory_resolved) or str(
        target_resolved
    ).startswith(str(directory_resolved) + os.sep)


def safe_extract_zip(zip_path: Path, extract_dir: Path) -> Dict[str, Any]:
    """
    Extract a ZIP archive defensively.

    Rejected members: absolute paths, parent traversal, symlinks / special files.
    Returns a report dictionary; never raises on a single bad member.
    """
    zip_path = Path(zip_path)
    extract_dir = Path(extract_dir)
    extract_dir.mkdir(parents=True, exist_ok=True)

    report: Dict[str, Any] = {
        "zip_path": str(zip_path),
        "zip_size_bytes": zip_path.stat().st_size,
        "extract_dir": str(extract_dir),
        "members_total": 0,
        "members_extracted": 0,
        "members_skipped_existing": 0,
        "members_rejected": 0,
        "rejected": [],
        "directories_created": 0,
        "elapsed_s": 0.0,
    }

    started = time.time()

    if not zipfile.is_zipfile(zip_path):
        raise ValueError(f"Not a valid ZIP archive: {zip_path}")

    with zipfile.ZipFile(zip_path) as archive:
        members = archive.infolist()
        report["members_total"] = len(members)

        for member in members:
            name = member.filename

            # --- structural rejections -------------------------------------
            normalised = name.replace("\\", "/")
            is_absolute = normalised.startswith("/") or bool(re.match(r"^[A-Za-z]:", normalised))
            if is_absolute:
                report["members_rejected"] += 1
                report["rejected"].append({"name": name, "reason": "ABSOLUTE_PATH"})
                continue
            if ".." in Path(normalised).parts:
                report["members_rejected"] += 1
                report["rejected"].append({"name": name, "reason": "PARENT_TRAVERSAL"})
                continue

            # Unix mode is stored in the high 16 bits of external_attr.
            # 0o100000 = regular file, 0o040000 = directory, 0 = created by a tool
            # that did not record a unix mode (common on Windows).
            unix_mode = member.external_attr >> 16
            file_type = unix_mode & 0o170000
            if unix_mode and file_type not in (0o100000, 0o040000, 0):
                report["members_rejected"] += 1
                report["rejected"].append({"name": name, "reason": "NOT_A_REGULAR_FILE"})
                continue

            destination = extract_dir / name
            if not _is_within_directory(extract_dir, destination):
                report["members_rejected"] += 1
                report["rejected"].append({"name": name, "reason": "ESCAPES_DESTINATION"})
                continue

            if member.is_dir():
                destination.mkdir(parents=True, exist_ok=True)
                report["directories_created"] += 1
                continue

            if destination.exists() and destination.stat().st_size == member.file_size:
                report["members_skipped_existing"] += 1
                continue

            destination.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(member) as source, open(destination, "wb") as sink:
                shutil.copyfileobj(source, sink, length=1024 * 1024)
            report["members_extracted"] += 1

    report["elapsed_s"] = round(time.time() - started, 2)
    return report


ZIP_PATH_RESOLVED = resolve_zip_path(CFG)
log(f"ZIP archive     : {ZIP_PATH_RESOLVED} ({ZIP_PATH_RESOLVED.stat().st_size:,} bytes)")

EXTRACTION_REPORT = safe_extract_zip(ZIP_PATH_RESOLVED, CFG.EXTRACT_DIR)

print("=" * 78)
print("ZIP EXTRACTION")
print("=" * 78)
print(f"Archive              : {EXTRACTION_REPORT['zip_path']}")
print(f"Archive size         : {EXTRACTION_REPORT['zip_size_bytes']:,} bytes")
print(f"Destination          : {EXTRACTION_REPORT['extract_dir']}")
print(f"Members in archive   : {EXTRACTION_REPORT['members_total']}")
print(f"Extracted            : {EXTRACTION_REPORT['members_extracted']}")
print(f"Already present      : {EXTRACTION_REPORT['members_skipped_existing']}")
print(f"Directories          : {EXTRACTION_REPORT['directories_created']}")
print(f"Rejected (unsafe)    : {EXTRACTION_REPORT['members_rejected']}")
for _item in EXTRACTION_REPORT["rejected"][:20]:
    print(f"   REJECTED {_item['reason']:20s} {_item['name']}")
print(f"Elapsed              : {EXTRACTION_REPORT['elapsed_s']} s")
print("=" * 78)

log(
    "ZIP extraction done: "
    f"{EXTRACTION_REPORT['members_extracted']} extracted, "
    f"{EXTRACTION_REPORT['members_skipped_existing']} reused, "
    f"{EXTRACTION_REPORT['members_rejected']} rejected"
)

## 6 — KYC inventory and controlled filename matching

### Normalisation rules (and only these)

| Rule | Example |
|---|---|
| Unicode NFKD + accent removal | `JUSTIFICATIF IDENTITÉ.PDF` → `JUSTIFICATIF IDENTITE.PDF` |
| Uppercase | `fatca.pdf` → `FATCA.PDF` |
| `_` and `-` → space | `JUSTIFICATIF_IDENTITE.PDF` → `JUSTIFICATIF IDENTITE.PDF` |
| Whitespace collapse + trim | `JUSTIFICATIF␣␣IDENTITE.PDF` → `JUSTIFICATIF IDENTITE.PDF` |
| Duplicate suffix stripped | `FATCA (1).PDF`, `FATCA - COPIE.PDF` → `FATCA.PDF` |
| Extension normalised | `.pdf` → `.PDF` |
| Explicit alias table | `CARTON SIGNATURE.PDF` → `CARTON SIGNATUTE.PDF` |

Matching is then **exact equality** against the canonical list. There is
deliberately **no fuzzy / edit-distance matching**: mapping an unrelated PDF onto
one of the five regulated document categories is a worse failure than reporting
it as unmatched.

`CARTON SIGNATUTE.PDF` — the historical business spelling — remains the canonical
key. The corrected spelling is accepted only through the explicit alias table.

### Duplicates

If several physical files normalise to the same logical type, **all** of them are
recorded, the situation is flagged, and behaviour is configurable
(`CFG.PROCESS_DUPLICATES`). For a RAW diagnostic run the default is to process
every physical file rather than silently pick one.

In [ ]:
# Recognised duplicate/copy suffixes, stripped before matching.
# Applied BEFORE '_' and '-' are turned into spaces, so the separator is still
# visible and "FATCA COPIE INTEGRALE.PDF" is NOT mistaken for a copy of FATCA.
DUPLICATE_SUFFIX_RE = re.compile(
    r"\s*(?:"
    r"\(\s*\d{1,3}\s*\)"                      # (1) (2) ...
    r"|\[\s*\d{1,3}\s*\]"                     # [1] [2] ...
    r"|\(\s*(?:COPIE|COPY)\s*\d{0,3}\s*\)"    # (COPIE) (COPY 2)
    r"|-\s*(?:COPIE|COPY)(?:\s*\d{0,3})?"     # - COPIE, -COPY 2
    r"|_\s*\d{1,3}"                           # _1 _2
    r")\s*$",
    re.IGNORECASE,
)


def strip_accents(text: str) -> str:
    """NFKD decomposition, combining marks dropped. Latin script only in practice."""
    decomposed = unicodedata.normalize("NFKD", text)
    return "".join(char for char in decomposed if not unicodedata.combining(char))


def normalize_filename(filename: str) -> Tuple[str, Optional[str]]:
    """
    Apply the controlled normalisation rules.

    Returns (normalised_name, duplicate_marker) where duplicate_marker is the
    recognised copy suffix (e.g. "(1)") or None.
    """
    raw = str(filename)
    stem = Path(raw).stem
    suffix = Path(raw).suffix

    # 1. accents, non-breaking spaces, case, whitespace
    stem = strip_accents(stem).replace(" ", " ")
    stem = re.sub(r"\s+", " ", stem).strip().upper()

    # 2. duplicate/copy suffix, while '_' and '-' are still distinguishable
    duplicate_marker = None
    match = DUPLICATE_SUFFIX_RE.search(stem)
    if match:
        duplicate_marker = match.group(0).strip()
        stem = DUPLICATE_SUFFIX_RE.sub("", stem).strip()

    # 3. separators -> space, collapse again
    stem = stem.replace("_", " ").replace("-", " ")
    stem = re.sub(r"\s+", " ", stem).strip()

    normalised_suffix = suffix.upper() if suffix else ""
    return f"{stem}{normalised_suffix}", duplicate_marker


def match_logical_document(filename: str) -> Tuple[Optional[str], str, Optional[str]]:
    """
    Map a physical filename onto one of the five canonical KYC document types.

    Returns (logical_type_or_None, normalised_name, duplicate_marker).
    Exact match after normalisation, plus the explicit alias table. Nothing else.
    """
    normalised, duplicate_marker = normalize_filename(filename)

    if normalised in TARGET_DOCUMENTS:
        return normalised, normalised, duplicate_marker

    aliased = DOCUMENT_ALIASES.get(normalised)
    if aliased in TARGET_DOCUMENTS:
        return aliased, normalised, duplicate_marker

    return None, normalised, duplicate_marker


# ---------------------------------------------------------------- self-test --
_MATCHING_CASES = [
    ("FATCA.PDF", "FATCA.PDF"),
    ("fatca.pdf", "FATCA.PDF"),
    ("FATCA (1).PDF", "FATCA.PDF"),
    ("FATCA (2).pdf", "FATCA.PDF"),
    ("JUSTIFICATIF IDENTITE.pdf", "JUSTIFICATIF IDENTITE.PDF"),
    ("JUSTIFICATIF  IDENTITE.PDF", "JUSTIFICATIF IDENTITE.PDF"),
    ("JUSTIFICATIF_IDENTITE.PDF", "JUSTIFICATIF IDENTITE.PDF"),
    ("Justificatif Identité.pdf", "JUSTIFICATIF IDENTITE.PDF"),
    ("CARTON SIGNATUTE.PDF", "CARTON SIGNATUTE.PDF"),
    ("CARTON SIGNATURE.PDF", "CARTON SIGNATUTE.PDF"),
    ("CONVENTION COMPTE.PDF", "CONVENTION COMPTE.PDF"),
    ("JUSTIFICATIF DOMICILE - COPIE.PDF", "JUSTIFICATIF DOMICILE.PDF"),
    ("RIB.PDF", None),
    ("FATCA_ANNEXE_TECHNIQUE.PDF", None),
    ("CONTRAT DE TRAVAIL.PDF", None),
    ("scan001.pdf", None),
]

print("Filename matching self-test")
print("-" * 78)
_failures = 0
for _candidate, _expected in _MATCHING_CASES:
    _logical, _normalised, _marker = match_logical_document(_candidate)
    _ok = _logical == _expected
    _failures += 0 if _ok else 1
    print(
        f"  {'PASS' if _ok else 'FAIL'}  {_candidate:38s} -> "
        f"{str(_logical):30s} (norm={_normalised}, dup={_marker})"
    )
print("-" * 78)
if _failures:
    raise AssertionError(f"{_failures} filename-matching self-test(s) failed.")
print(f"All {len(_MATCHING_CASES)} matching cases behave as specified.")

In [ ]:
def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    """Streamed SHA-256 (same helper as dom.ipynb)."""
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def find_customer_root(extract_dir: Path) -> Path:
    """
    Locate the directory that actually contains the customer folders.

    Some archives wrap everything in a single top-level folder
    (kyc_documents/123456/...). One level of wrapping is unwrapped; deeper or
    ambiguous layouts are left alone and reported.
    """
    extract_dir = Path(extract_dir)
    entries = [item for item in extract_dir.iterdir() if not item.name.startswith(".")]
    directories = [item for item in entries if item.is_dir()]
    files = [item for item in entries if item.is_file()]

    if len(directories) == 1 and not files:
        inner = directories[0]
        inner_directories = [
            item for item in inner.iterdir()
            if item.is_dir() and not item.name.startswith(".")
        ]
        if inner_directories:
            log(f"Single wrapper folder detected, descending into: {inner.name}")
            return inner

    return extract_dir


def build_inventory(customer_root: Path) -> pd.DataFrame:
    """
    Walk every customer folder and produce the inventory table.

    One row per (customer_id, expected_document_type) — always five rows per
    customer, so missing documents are explicit rather than absent.
    """
    customer_root = Path(customer_root)
    rows: List[Dict[str, Any]] = []
    unmatched_rows: List[Dict[str, Any]] = []

    customer_directories = sorted(
        (item for item in customer_root.iterdir()
         if item.is_dir() and not item.name.startswith(".")),
        key=lambda item: item.name,
    )

    if not customer_directories:
        raise RuntimeError(
            f"No customer directory found under {customer_root}. "
            "Check the archive layout."
        )

    for customer_directory in customer_directories:
        customer_id = customer_directory.name.strip()

        # Every file directly inside the customer folder, plus one nested level,
        # because some exports add an intermediate subfolder.
        candidate_files = [
            item for item in customer_directory.rglob("*")
            if item.is_file() and not item.name.startswith(".")
        ]

        grouped: Dict[str, List[Path]] = defaultdict(list)
        for file_path in sorted(candidate_files, key=lambda item: item.name):
            logical, normalised, marker = match_logical_document(file_path.name)
            if logical is None:
                unmatched_rows.append({
                    "customer_id": customer_id,
                    "filename": file_path.name,
                    "normalized_filename": normalised,
                    "duplicate_marker": marker,
                    "relative_path": str(file_path.relative_to(customer_root)),
                    "file_size_bytes": file_path.stat().st_size,
                })
                continue
            grouped[logical].append(file_path)

        for expected_type in TARGET_DOCUMENTS:
            matches = grouped.get(expected_type, [])
            if matches:
                primary = matches[0]
                rows.append({
                    "customer_id": customer_id,
                    "expected_document_type": expected_type,
                    "exists": True,
                    "matched_filename": primary.name,
                    "full_path": str(primary),
                    "duplicate_count": len(matches),
                    "duplicate_filenames": "|".join(item.name for item in matches),
                    "duplicate_full_paths": "|".join(str(item) for item in matches),
                    "file_size_bytes": primary.stat().st_size,
                    "is_ambiguous": len(matches) > 1,
                })
            else:
                rows.append({
                    "customer_id": customer_id,
                    "expected_document_type": expected_type,
                    "exists": False,
                    "matched_filename": None,
                    "full_path": None,
                    "duplicate_count": 0,
                    "duplicate_filenames": "",
                    "duplicate_full_paths": "",
                    "file_size_bytes": 0,
                    "is_ambiguous": False,
                })

    inventory = pd.DataFrame(rows)
    inventory.attrs["unmatched"] = unmatched_rows
    inventory.attrs["customer_root"] = str(customer_root)
    return inventory


CUSTOMER_ROOT = find_customer_root(CFG.EXTRACT_DIR)
INVENTORY_DF = build_inventory(CUSTOMER_ROOT)
UNMATCHED_FILES = INVENTORY_DF.attrs.get("unmatched", [])

CUSTOMER_IDS = sorted(INVENTORY_DF["customer_id"].unique().tolist())

print("=" * 78)
print("KYC INVENTORY")
print("=" * 78)
print(f"Customer root        : {CUSTOMER_ROOT}")
print(f"Customers found      : {len(CUSTOMER_IDS)}")
print(f"Inventory rows       : {len(INVENTORY_DF)}  (= customers x {len(TARGET_DOCUMENTS)} expected types)")
print(f"Target PDFs found    : {int(INVENTORY_DF['exists'].sum())}")
print(f"Target PDFs missing  : {int((~INVENTORY_DF['exists']).sum())}")
print(f"Ambiguous duplicates : {int(INVENTORY_DF['is_ambiguous'].sum())}")
print(f"Unmatched files      : {len(UNMATCHED_FILES)} (not one of the five target types)")
print("-" * 78)
print("Coverage per document type:")
for _expected in TARGET_DOCUMENTS:
    _subset = INVENTORY_DF[INVENTORY_DF["expected_document_type"] == _expected]
    _found = int(_subset["exists"].sum())
    _duplicates = int(_subset["is_ambiguous"].sum())
    _pct = (100.0 * _found / len(_subset)) if len(_subset) else 0.0
    print(f"  {_expected:28s} found={_found:5d}/{len(_subset):<5d} ({_pct:5.1f}%)  ambiguous={_duplicates}")

if INVENTORY_DF["is_ambiguous"].any():
    print("-" * 78)
    print("Ambiguous duplicates (ALL physical files are recorded, none discarded):")
    for _, _row in INVENTORY_DF[INVENTORY_DF["is_ambiguous"]].head(20).iterrows():
        print(f"  {_row['customer_id']:>12s} | {_row['expected_document_type']:28s} | {_row['duplicate_filenames']}")

if UNMATCHED_FILES:
    print("-" * 78)
    print("Sample of unmatched files (deliberately NOT force-mapped):")
    for _item in UNMATCHED_FILES[:15]:
        print(f"  {_item['customer_id']:>12s} | {_item['filename']}")
print("=" * 78)

log(
    f"Inventory built: {len(CUSTOMER_IDS)} customers, "
    f"{int(INVENTORY_DF['exists'].sum())} target PDFs, "
    f"{int((~INVENTORY_DF['exists']).sum())} missing, "
    f"{int(INVENTORY_DF['is_ambiguous'].sum())} ambiguous"
)

## 7 — Inventory reports

Written to `outputs/inventory/`. The JSON carries the same rows plus the
unmatched-file list and the extraction report, so the inventory is auditable
without re-running anything.

In [ ]:
def json_default(value: Any) -> Any:
    """
    JSON fallback encoder.

    NumPy scalars, NumPy arrays, torch tensors, Path, datetime and sets all appear
    in these records; none of them is JSON-serialisable by default.
    """
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().tolist()
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, (datetime,)):
        return value.isoformat(timespec="seconds")
    if isinstance(value, (set, frozenset)):
        return sorted(value, key=str)
    return str(value)


def to_jsonable(obj: Any) -> Any:
    """Recursively convert a structure into plain JSON-safe Python."""
    if obj is None or isinstance(obj, (bool, int, float, str)):
        if isinstance(obj, float) and (math.isnan(obj) or math.isinf(obj)):
            return None
        return obj
    if isinstance(obj, dict):
        return {str(key): to_jsonable(val) for key, val in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [to_jsonable(item) for item in obj]
    return json_default(obj)


def write_json(path: Path, payload: Any) -> Path:
    """Atomic JSON write: temp file then rename, so an interrupt cannot truncate."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    with open(temporary, "w", encoding="utf-8") as handle:
        json.dump(to_jsonable(payload), handle, ensure_ascii=False, indent=2, default=json_default)
    os.replace(temporary, path)
    return path


INVENTORY_CSV_PATH = CFG.INVENTORY_DIR / "kyc_document_inventory.csv"
INVENTORY_JSON_PATH = CFG.INVENTORY_DIR / "kyc_document_inventory.json"

INVENTORY_COLUMNS = [
    "customer_id",
    "expected_document_type",
    "exists",
    "matched_filename",
    "full_path",
    "duplicate_count",
    "duplicate_filenames",
    "file_size_bytes",
    "is_ambiguous",
    "duplicate_full_paths",
]

INVENTORY_DF[INVENTORY_COLUMNS].to_csv(INVENTORY_CSV_PATH, index=False, encoding="utf-8-sig")

write_json(INVENTORY_JSON_PATH, {
    "generated_at": datetime.now().isoformat(timespec="seconds"),
    "pipeline_version": PIPELINE_VERSION,
    "customer_root": str(CUSTOMER_ROOT),
    "zip_extraction": EXTRACTION_REPORT,
    "target_documents": TARGET_DOCUMENTS,
    "document_aliases": DOCUMENT_ALIASES,
    "summary": {
        "customers": len(CUSTOMER_IDS),
        "rows": int(len(INVENTORY_DF)),
        "target_pdfs_found": int(INVENTORY_DF["exists"].sum()),
        "target_pdfs_missing": int((~INVENTORY_DF["exists"]).sum()),
        "ambiguous_duplicates": int(INVENTORY_DF["is_ambiguous"].sum()),
        "unmatched_files": len(UNMATCHED_FILES),
    },
    "inventory": INVENTORY_DF[INVENTORY_COLUMNS].to_dict(orient="records"),
    "unmatched_files": UNMATCHED_FILES,
})

print(f"Inventory CSV  : {INVENTORY_CSV_PATH}")
print(f"Inventory JSON : {INVENTORY_JSON_PATH}")
print()
print("First rows:")
print(INVENTORY_DF[["customer_id", "expected_document_type", "exists",
                    "matched_filename", "duplicate_count", "file_size_bytes"]].head(12).to_string(index=False))

log(f"Inventory reports written to {CFG.INVENTORY_DIR}")

## 8 — PDF and image utilities

Adapted from the proven `dom.ipynb` helpers (`pdf_to_pages`, `resize_image`,
`white_ratio`, `is_blank`), with one deliberate change.

**Change: per-page rendering instead of whole-PDF materialisation.**
`dom.ipynb` rendered every page of a PDF into a list of PIL images before doing
any inference. For RAW OCR that is the wrong shape:

* every page is independently checkpointed, so a whole-PDF render is wasted work
  as soon as a run is resumed;
* a 2200-px HD page is ≈ 10 MB of RGB in RAM — holding a 30-page file costs
  300 MB for no reason;
* per-page timing becomes exact instead of amortised.

`render_pdf_page()` is therefore the primitive, and `pdf_to_pages()` is kept as a
thin compatibility wrapper for manual inspection.

The PDF handle is opened and **closed** for every page (`try/finally`), matching
`dom.ipynb`. `page_count` is always read dynamically — never assumed.

In [ ]:
def resize_image(image: Image.Image, max_side: int) -> Image.Image:
    """Aspect-preserving downscale. Never upscales (same contract as dom.ipynb)."""
    width, height = image.size
    if max(width, height) <= max_side:
        return image
    ratio = max_side / float(max(width, height))
    new_size = (max(1, int(round(width * ratio))), max(1, int(round(height * ratio))))
    return image.resize(new_size, RESAMPLE_LANCZOS)


def white_ratio(image: Image.Image) -> float:
    """Fraction of near-white pixels (same heuristic as dom.ipynb)."""
    array = np.array(image.convert("L"))
    if array.size == 0:
        return 1.0
    return float((array > 245).sum() / array.size)


def ink_ratio(image: Image.Image, threshold: int = 200) -> float:
    """Fraction of dark pixels. Used for blank detection and fallback triggers."""
    array = np.array(image.convert("L"))
    if array.size == 0:
        return 0.0
    return float((array < threshold).sum() / array.size)


def is_blank(image: Image.Image, threshold: float = None) -> bool:
    threshold = CFG.BLANK_THRESHOLD if threshold is None else threshold
    return white_ratio(image) >= threshold


def pdf_page_count(path: Path) -> int:
    """Dynamic page count. Never assume a fixed number of pages."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"PDF not found: {path}")
    if path.stat().st_size == 0:
        raise ValueError(f"Empty PDF file: {path}")
    document = fitz.open(str(path))
    try:
        return int(document.page_count)
    finally:
        document.close()


def render_pdf_page(
    path: Path,
    page_index: int,
    zoom: float,
    max_side: int,
) -> Dict[str, Any]:
    """
    Render a single PDF page to an RGB PIL image.

    Returns the image plus the measurements needed for performance attribution:
    native render size, resized size, white ratio, and render time.
    """
    path = Path(path)
    started = time.time()

    document = fitz.open(str(path))
    try:
        page_count = int(document.page_count)
        if page_count <= 0:
            raise ValueError(f"PyMuPDF reports no page in {path.name}")
        if not 0 <= int(page_index) < page_count:
            raise IndexError(
                f"Page index {page_index} out of range for {path.name} ({page_count} pages)"
            )

        page = document.load_page(int(page_index))
        pixmap = page.get_pixmap(matrix=fitz.Matrix(zoom, zoom), alpha=False)

        if pixmap.width <= 0 or pixmap.height <= 0 or not pixmap.samples:
            raise ValueError(f"Empty render: {path.name}, page {int(page_index) + 1}")

        # PyMuPDF may hand back 1 (gray) or 3 (RGB) components depending on the
        # source colourspace; normalise to RGB the way dom.ipynb assumes.
        mode = "RGB" if pixmap.n == 3 else ("L" if pixmap.n == 1 else "RGBA")
        image = Image.frombytes(mode, (pixmap.width, pixmap.height), pixmap.samples)
        if image.mode != "RGB":
            image = image.convert("RGB")

        native_width, native_height = image.size
        page_rotation = int(getattr(page, "rotation", 0) or 0)
    finally:
        document.close()

    image = resize_image(image, max_side=max_side)

    return {
        "image": image,
        "page_index": int(page_index),
        "page_number": int(page_index) + 1,
        "page_count": page_count,
        "zoom": float(zoom),
        "max_side": int(max_side),
        # render_* and original_* are the same measurement under the two names
        # used in the specification; both are exposed so neither report breaks.
        "render_width": int(native_width),
        "render_height": int(native_height),
        "original_width": int(native_width),
        "original_height": int(native_height),
        "processed_width": int(image.width),
        "processed_height": int(image.height),
        "page_rotation": page_rotation,
        "white_ratio": round(white_ratio(image), 6),
        "render_time_s": round(time.time() - started, 4),
    }


def pdf_to_pages(
    path: Path,
    zoom: float = None,
    max_side: int = None,
) -> List[Dict[str, Any]]:
    """
    Compatibility wrapper over render_pdf_page() for manual inspection.

    The production loop renders page by page; this materialises the whole PDF and
    should only be used interactively on a small file.
    """
    zoom = CFG.RENDER_ZOOM_STANDARD if zoom is None else zoom
    max_side = CFG.IMAGE_MAX_SIZE_STANDARD if max_side is None else max_side

    total = pdf_page_count(path)
    pages = [render_pdf_page(path, index, zoom, max_side) for index in range(total)]

    if len(pages) != total:
        raise RuntimeError(
            f"Incomplete conversion of {Path(path).name}: {len(pages)} image(s) for {total} page(s)"
        )
    return pages


print("PDF / image utilities ready:")
print("  sha256_file, resize_image, white_ratio, ink_ratio, is_blank,")
print("  pdf_page_count, render_pdf_page, pdf_to_pages")

## 9 — Generic conservative preprocessing

The document-specific crops of `dom.ipynb` (`crops_planche_permis`,
`detecter_frontiere_documents`) were tuned to one business form and are **not**
carried over. What replaces them is generic and conservative.

### Governing rule

> **Never destroy information in the name of preprocessing.**

Concretely: faint handwriting, Arabic diacritics, MRZ `<` glyphs, thin digits,
stamps and check marks must survive. Every step is therefore either
*information-preserving by construction* or *guarded*:

| Step | Default | Rationale |
|---|---|---|
| `ENABLE_AUTO_CROP` | on | Removes only empty margins, keeps a 1.2 % safety pad, refuses to crop unless it saves ≥ 8 % of the area and keeps ≥ 25 % of the page. An ID card scanned in the middle of an A4 gains real resolution from this. |
| `ENABLE_DESKEW` | on | Projection-profile angle search over ±3°, applied only above 0.4°, bicubic with white fill. Below 0.4° rotation is pure resampling loss. |
| `ENABLE_CONTRAST` | on | Percentile stretch (p2–p98) applied **only** when the dynamic range is already compressed (`< 200/255`). A well-exposed scan is left untouched. |
| `ENABLE_SHARPEN` | on | Mild unsharp mask (r=1.2, 60 %, threshold 3). The threshold keeps flat paper noise from being amplified. |
| `ENABLE_DENOISE` | **off** | Median/rank filters erase 1-px strokes: Arabic diacritics, accents and MRZ separators. Available, disabled, and deliberately so. |
| `ENABLE_ORIENTATION` | **off** | See below. |
| Binarisation | **absent** | Thresholding a faint blue-ink signature or a grey stamp deletes it outright. Qwen is a vision model, not a 1990s OCR engine — it does not need binary input. |

### Global guard

After the chain runs, the ink ratio of the result is compared to the original.
If more than `PREPROCESS_MAX_INK_LOSS` (25 %) of the ink has disappeared, the
**original render is restored** and the reason is recorded. Preprocessing can
therefore never silently degrade a page.

### Why orientation correction is off by default

A projection profile reliably detects a **90°** error (text lines run the wrong
way) but is mathematically incapable of detecting a **180°** flip — upside-down
text has the same horizontal line structure. Applying a wrong 90° rotation is
catastrophic, so for a *baseline* measurement the correct experiment is to first
find out whether Qwen already handles rotated pages natively. The detector is
implemented and audited on every page (`orientation_suggestion` is recorded even
when nothing is applied); turn `ENABLE_ORIENTATION = True` on only if the
diagnostic run shows rotated pages coming back garbled.

In [ ]:
def _grayscale_array(image: Image.Image, max_side: int = 900) -> np.ndarray:
    """Downsampled grayscale array used by the cheap geometric estimators."""
    small = resize_image(image, max_side=max_side).convert("L")
    return np.asarray(small, dtype=np.uint8)


def detect_ink_bbox(
    image: Image.Image,
    ink_threshold: int = None,
    margin_fraction: float = None,
) -> Optional[Tuple[int, int, int, int]]:
    """
    Bounding box of the inked area, padded with a safety margin.

    Returns None when the page is empty or when ink already spans the page.
    """
    ink_threshold = CFG.AUTO_CROP_INK_THRESHOLD if ink_threshold is None else ink_threshold
    margin_fraction = CFG.AUTO_CROP_MARGIN_FRACTION if margin_fraction is None else margin_fraction

    array = np.asarray(image.convert("L"), dtype=np.uint8)
    mask = array < ink_threshold
    if not mask.any():
        return None

    rows = np.where(mask.any(axis=1))[0]
    columns = np.where(mask.any(axis=0))[0]
    top, bottom = int(rows[0]), int(rows[-1]) + 1
    left, right = int(columns[0]), int(columns[-1]) + 1

    pad_y = int(round(array.shape[0] * margin_fraction))
    pad_x = int(round(array.shape[1] * margin_fraction))
    top = max(0, top - pad_y)
    left = max(0, left - pad_x)
    bottom = min(array.shape[0], bottom + pad_y)
    right = min(array.shape[1], right + pad_x)

    if right <= left or bottom <= top:
        return None
    return (left, top, right, bottom)


def auto_crop_margins(image: Image.Image) -> Tuple[Image.Image, Dict[str, Any]]:
    """Crop empty margins, but only when the gain is real and the page survives."""
    info: Dict[str, Any] = {"applied": False, "bbox": None, "area_gain": 0.0, "reason": None}

    bbox = detect_ink_bbox(image)
    if bbox is None:
        info["reason"] = "NO_INK_OR_FULL_PAGE"
        return image, info

    left, top, right, bottom = bbox
    original_area = float(image.width * image.height)
    cropped_area = float((right - left) * (bottom - top))
    if original_area <= 0:
        info["reason"] = "DEGENERATE_IMAGE"
        return image, info

    kept_fraction = cropped_area / original_area
    gain = 1.0 - kept_fraction

    if gain < CFG.AUTO_CROP_MIN_GAIN:
        info["reason"] = "GAIN_TOO_SMALL"
        info["area_gain"] = round(gain, 4)
        return image, info
    if kept_fraction < CFG.AUTO_CROP_MIN_KEPT_FRACTION:
        info["reason"] = "WOULD_KEEP_TOO_LITTLE"
        info["area_gain"] = round(gain, 4)
        return image, info

    info.update({"applied": True, "bbox": [left, top, right, bottom], "area_gain": round(gain, 4)})
    return image.crop((left, top, right, bottom)), info


def estimate_skew_angle(image: Image.Image) -> float:
    """
    Projection-profile skew estimate.

    The horizontal projection of a correctly aligned text page has high variance
    (dense text rows alternating with empty inter-line gaps). The angle that
    maximises that variance is the deskew angle.
    """
    array = _grayscale_array(image)
    ink = (array < CFG.AUTO_CROP_INK_THRESHOLD).astype(np.float32)
    if ink.sum() < 50:
        return 0.0

    base = Image.fromarray((ink * 255).astype(np.uint8))
    best_angle, best_score = 0.0, -1.0

    steps = int(round(2 * CFG.DESKEW_MAX_ANGLE / CFG.DESKEW_STEP)) + 1
    for step in range(steps):
        angle = -CFG.DESKEW_MAX_ANGLE + step * CFG.DESKEW_STEP
        if abs(angle) < 1e-9:
            rotated = np.asarray(base, dtype=np.float32) / 255.0
        else:
            rotated = np.asarray(
                base.rotate(angle, resample=RESAMPLE_BICUBIC, fillcolor=0),
                dtype=np.float32,
            ) / 255.0
        profile = rotated.sum(axis=1)
        score = float(np.var(profile))
        if score > best_score:
            best_score, best_angle = score, angle

    return float(best_angle)


def deskew(image: Image.Image) -> Tuple[Image.Image, Dict[str, Any]]:
    """Rotate by the estimated skew only when the angle is worth the resampling."""
    info: Dict[str, Any] = {"applied": False, "angle": 0.0}
    angle = estimate_skew_angle(image)
    info["angle"] = round(angle, 3)

    if abs(angle) < CFG.DESKEW_MIN_ANGLE_APPLY:
        info["reason"] = "ANGLE_BELOW_THRESHOLD"
        return image, info

    rotated = image.rotate(
        angle, resample=RESAMPLE_BICUBIC, expand=True, fillcolor=(255, 255, 255)
    )
    info["applied"] = True
    return rotated, info


def enhance_contrast(image: Image.Image) -> Tuple[Image.Image, Dict[str, Any]]:
    """
    Percentile stretch, applied only to a flat / washed-out scan.

    Clipping is done at p2/p98 and the mapping is linear, so mid-tones (faint
    pencil, light blue ink) are lifted rather than removed.
    """
    info: Dict[str, Any] = {"applied": False, "p_low": None, "p_high": None}

    luminance = np.asarray(image.convert("L"), dtype=np.uint8)
    if luminance.size == 0:
        return image, info

    p_low = float(np.percentile(luminance, CFG.CONTRAST_PERCENTILE_LOW))
    p_high = float(np.percentile(luminance, CFG.CONTRAST_PERCENTILE_HIGH))
    info["p_low"], info["p_high"] = round(p_low, 1), round(p_high, 1)

    span = p_high - p_low
    if span >= CFG.CONTRAST_RANGE_TRIGGER or span < 10:
        info["reason"] = "RANGE_ALREADY_WIDE" if span >= CFG.CONTRAST_RANGE_TRIGGER else "RANGE_TOO_NARROW"
        return image, info

    array = np.asarray(image.convert("RGB"), dtype=np.float32)
    stretched = (array - p_low) * (255.0 / span)
    stretched = np.clip(stretched, 0, 255).astype(np.uint8)
    info["applied"] = True
    return Image.fromarray(stretched, mode="RGB"), info


def mild_sharpen(image: Image.Image) -> Tuple[Image.Image, Dict[str, Any]]:
    """Unsharp mask with a threshold, so flat paper grain is not amplified."""
    sharpened = image.filter(
        ImageFilter.UnsharpMask(
            radius=CFG.SHARPEN_RADIUS,
            percent=CFG.SHARPEN_PERCENT,
            threshold=CFG.SHARPEN_THRESHOLD,
        )
    )
    return sharpened, {"applied": True, "radius": CFG.SHARPEN_RADIUS, "percent": CFG.SHARPEN_PERCENT}


def mild_denoise(image: Image.Image) -> Tuple[Image.Image, Dict[str, Any]]:
    """
    3x3 median filter. DISABLED by default: it erases 1-pixel strokes
    (Arabic diacritics, accents, MRZ separators, thin digits).
    """
    return image.filter(ImageFilter.MedianFilter(size=3)), {"applied": True, "kernel": 3}


def detect_orientation(image: Image.Image) -> Dict[str, Any]:
    """
    Detect a 90-degree orientation error from projection-profile variance.

    Text lines produce a high-variance projection along the axis perpendicular to
    them. If the vertical projection is far more structured than the horizontal
    one, the page is rotated by 90 degrees.

    LIMITATION, by construction: a 180-degree flip is undetectable this way.
    """
    array = _grayscale_array(image)
    ink = (array < CFG.AUTO_CROP_INK_THRESHOLD).astype(np.float32)
    result: Dict[str, Any] = {
        "suggested_rotation": 0,
        "horizontal_variance": 0.0,
        "vertical_variance": 0.0,
        "evidence_ratio": 1.0,
        "note": "180-degree flips are not detectable by projection profiles",
    }
    if ink.sum() < 50:
        return result

    horizontal = float(np.var(ink.sum(axis=1)))  # profile across rows
    vertical = float(np.var(ink.sum(axis=0)))    # profile across columns
    result["horizontal_variance"] = round(horizontal, 3)
    result["vertical_variance"] = round(vertical, 3)

    if horizontal <= 0 and vertical <= 0:
        return result

    if vertical > horizontal * CFG.ORIENTATION_EVIDENCE_RATIO:
        result["suggested_rotation"] = 90
        result["evidence_ratio"] = round(vertical / max(horizontal, 1e-6), 2)
    else:
        result["evidence_ratio"] = round(horizontal / max(vertical, 1e-6), 2)

    return result


def preprocess_image(image: Image.Image, cfg: PipelineConfig = None) -> Tuple[Image.Image, Dict[str, Any]]:
    """
    Conservative KYC preprocessing chain with a global information guard.

    Returns (image, report). The ORIGINAL image is always returned unchanged when
    preprocessing is disabled or when the guard fires.
    """
    cfg = cfg or CFG
    report: Dict[str, Any] = {
        "enabled": bool(cfg.ENABLE_PREPROCESSING),
        "steps": {},
        "reverted": False,
        "revert_reason": None,
        "ink_before": None,
        "ink_after": None,
        "size_before": [image.width, image.height],
        "size_after": [image.width, image.height],
        "preprocess_time_s": 0.0,
    }

    # Orientation is measured on every page even when it is not applied.
    orientation = detect_orientation(image)
    report["orientation"] = orientation

    if not cfg.ENABLE_PREPROCESSING:
        return image, report

    started = time.time()
    original = image
    ink_before = ink_ratio(original)
    report["ink_before"] = round(ink_before, 6)

    working = image

    if cfg.ENABLE_ORIENTATION and orientation["suggested_rotation"] == 90:
        working = working.rotate(-90, expand=True, fillcolor=(255, 255, 255))
        report["steps"]["orientation"] = {"applied": True, "rotation": -90}
    else:
        report["steps"]["orientation"] = {
            "applied": False,
            "suggested": orientation["suggested_rotation"],
        }

    if cfg.ENABLE_AUTO_CROP:
        working, info = auto_crop_margins(working)
        report["steps"]["auto_crop"] = info

    if cfg.ENABLE_DESKEW:
        working, info = deskew(working)
        report["steps"]["deskew"] = info

    if cfg.ENABLE_CONTRAST:
        working, info = enhance_contrast(working)
        report["steps"]["contrast"] = info

    if cfg.ENABLE_DENOISE:
        working, info = mild_denoise(working)
        report["steps"]["denoise"] = info

    if cfg.ENABLE_SHARPEN:
        working, info = mild_sharpen(working)
        report["steps"]["sharpen"] = info

    ink_after = ink_ratio(working)
    report["ink_after"] = round(ink_after, 6)

    # ---- global information guard -------------------------------------------
    # Auto-crop legitimately RAISES the ink ratio (less blank paper), so only a
    # LOSS is suspicious.
    if ink_before > 1e-6 and ink_after < ink_before * (1.0 - cfg.PREPROCESS_MAX_INK_LOSS):
        report["reverted"] = True
        report["revert_reason"] = (
            f"ink ratio dropped {ink_before:.5f} -> {ink_after:.5f} "
            f"(> {cfg.PREPROCESS_MAX_INK_LOSS:.0%} loss)"
        )
        report["preprocess_time_s"] = round(time.time() - started, 4)
        report["size_after"] = [original.width, original.height]
        return original, report

    report["size_after"] = [working.width, working.height]
    report["preprocess_time_s"] = round(time.time() - started, 4)
    return working, report


print("Preprocessing ready (conservative, guarded):")
print(f"  ENABLE_PREPROCESSING = {CFG.ENABLE_PREPROCESSING}")
print(f"  ENABLE_AUTO_CROP     = {CFG.ENABLE_AUTO_CROP}")
print(f"  ENABLE_DESKEW        = {CFG.ENABLE_DESKEW}")
print(f"  ENABLE_CONTRAST      = {CFG.ENABLE_CONTRAST}")
print(f"  ENABLE_SHARPEN       = {CFG.ENABLE_SHARPEN}")
print(f"  ENABLE_DENOISE       = {CFG.ENABLE_DENOISE}   (off: destroys thin strokes)")
print(f"  ENABLE_ORIENTATION   = {CFG.ENABLE_ORIENTATION}   (off: 90-deg only, measured but not applied)")
print(f"  guard: revert if ink loss > {CFG.PREPROCESS_MAX_INK_LOSS:.0%}")

## 10 — Model loading (FP8)

### What `FineGrainedFP8Config(dequantize=True)` actually does here

The checkpoint on disk is **already FP8** (`Qwen3.6-27B-FP8`): its linear weights
are stored as `float8_e4m3fn` tensors accompanied by per-block scale tensors
(fine-grained, typically 128×128 blocks, hence *FineGrained*).

`FineGrainedFP8Config` is the Transformers quantizer that knows how to read that
layout. The `dequantize=True` flag changes what happens next:

| flag | behaviour |
|---|---|
| `dequantize=False` | Keep `FP8Linear` modules; multiply in FP8 on the fly. Needs a matching kernel path and the right compute capability; this is what saves VRAM. |
| `dequantize=True` | Read the FP8 blocks **once at load time**, multiply by their scales, and materialise ordinary `nn.Linear` weights in `dtype` (here `torch.bfloat16`). |

So in this configuration FP8 is a **storage format on disk only**. The model that
ends up on the H100 is a plain bf16 model — which is exactly why
`dtype=torch.bfloat16` is passed alongside, and why it is not a contradiction.

Three consequences that matter for this notebook:

1. **Nothing is re-quantised.** The FP8 checkpoint is read, not compressed again.
   This is the failure mode the brief warns about, and it does not occur here.
2. **VRAM is bf16-sized**, ≈ 54 GB of weights for 27 B parameters. It fits on an
   80 GB H100 with room for activations and KV cache — but only just, which is
   why §11 checks for CPU offload explicitly.
3. **Decode speed is bf16 speed**, ≈ 25–40 tok/s. This is the number that governs
   page latency, and it is why `max_new_tokens` matters more than image size.

### Why this path is kept unchanged

`dequantize=False` would roughly halve the weight memory and could be faster, but
it depends on the installed Transformers version, the kernel availability and the
exact scale layout of this checkpoint. `dom.ipynb` **proved** the dequantised path
works in this environment. A RAW OCR baseline exists to measure OCR quality — it
is the wrong place to also gamble on a different quantisation path.

The cell below therefore uses the `dom.ipynb` load verbatim, and §11 prints the
evidence (`model.config.quantization_config`, parameter dtype, device map) needed
to decide later, with data, whether `dequantize=False` is worth trying.

`FP8Config` is imported through the same `try/except ImportError` as
`dom.ipynb`, which covers both the `transformers.integrations.finegrained_fp8`
and the top-level `transformers` location. The legacy `FP8Linear` import is
deliberately **not** used.

In [ ]:
if DEVICE != "cuda":
    raise RuntimeError("This pipeline requires a CUDA GPU (same constraint as dom.ipynb).")

if not Path(CFG.MODEL_PATH).exists():
    raise FileNotFoundError(
        f"Model path not found: {CFG.MODEL_PATH}\n"
        "This is the local Domino ModelHub path used by dom.ipynb. Adjust CFG.MODEL_PATH if it moved."
    )

# Same TF32 policy as dom.ipynb (re-applied here so this cell is self-sufficient).
torch.backends.cuda.matmul.allow_tf32 = True

print("Loading processor ...")
_t_processor = time.time()
processor = AutoProcessor.from_pretrained(
    CFG.MODEL_PATH,
    trust_remote_code=True,
    min_pixels=CFG.MIN_PIXELS,
    max_pixels=CFG.MAX_PIXELS,
)
processor.tokenizer.padding_side = "left"
PROCESSOR_LOAD_TIME_S = round(time.time() - _t_processor, 2)
print(f"Processor loaded in {PROCESSOR_LOAD_TIME_S}s")

# Same import contract as dom.ipynb: new location first, legacy fallback.
try:
    from transformers.integrations.finegrained_fp8 import FineGrainedFP8Config as FP8Config
    FP8_CONFIG_SOURCE = "transformers.integrations.finegrained_fp8"
except ImportError:
    from transformers import FineGrainedFP8Config as FP8Config
    FP8_CONFIG_SOURCE = "transformers (top level)"

print(f"FP8Config imported from: {FP8_CONFIG_SOURCE}")

print("Loading FP8 model (dequantised to bfloat16) ...")
_t_model = time.time()
model = AutoModelForImageTextToText.from_pretrained(
    CFG.MODEL_PATH,
    dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
    low_cpu_mem_usage=True,
    quantization_config=FP8Config(dequantize=True),
)
model.eval()
MODEL_LOAD_TIME_S = round(time.time() - _t_model, 2)

torch.cuda.synchronize()
print(f"Model loaded in {MODEL_LOAD_TIME_S}s")
print(f"VRAM allocated : {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"VRAM reserved  : {torch.cuda.memory_reserved() / 1e9:.2f} GB")

log(
    f"Model loaded: {CFG.MODEL_PATH} | processor={PROCESSOR_LOAD_TIME_S}s "
    f"| model={MODEL_LOAD_TIME_S}s | fp8_config_source={FP8_CONFIG_SOURCE}"
)

## 11 — Model / inference diagnostics

Three questions this cell must answer before a single page is processed:

1. **Is the model really on the H100?** `device_map="auto"` silently spills to CPU
   (and even to disk) when VRAM is short. A 27 B bf16 model with a handful of
   layers on CPU does not run "a bit slower" — it runs 10–50× slower, and that
   single fact explains most multi-hour OCR runs.
2. **What is the actual parameter dtype?** If the FP8 dequantisation did not
   happen as expected, this is where it shows.
3. **Where do inputs belong?** With `device_map="auto"` the correct target for
   `inputs.to(...)` is the device that holds the **input embeddings**, not a
   hard-coded `"cuda"`. On a single-GPU H100 the two are identical, so this is a
   no-op here — but it is correct under any device map, and it turns a silent
   cross-device error into an explicit diagnostic.

This is the one deviation from `dom.ipynb`'s `inputs.to(DEVICE)`, and it is
behaviour-preserving in the proven single-GPU case.

In [ ]:
def resolve_input_device(target_model) -> torch.device:
    """
    Device that input tensors must be moved to.

    With device_map="auto" the input_ids must land on the device holding the input
    embeddings; accelerate's hooks then move activations across devices. Falls
    back to the first parameter, then to the global DEVICE.
    """
    try:
        embeddings = target_model.get_input_embeddings()
    except Exception:
        embeddings = None

    if embeddings is not None:
        try:
            return next(embeddings.parameters()).device
        except (StopIteration, AttributeError):
            pass

    try:
        return next(target_model.parameters()).device
    except StopIteration:
        return torch.device(DEVICE)


def summarise_device_map(target_model) -> Dict[str, Any]:
    """Aggregate hf_device_map and detect CPU / disk offloading."""
    device_map = getattr(target_model, "hf_device_map", None)
    summary: Dict[str, Any] = {
        "has_device_map": device_map is not None,
        "devices": {},
        "cpu_modules": [],
        "disk_modules": [],
        "offloaded": False,
    }
    if not device_map:
        return summary

    counter: Dict[str, int] = defaultdict(int)
    for module_name, device in device_map.items():
        key = str(device)
        counter[key] += 1
        if key in ("cpu", "meta"):
            summary["cpu_modules"].append(module_name)
        elif key == "disk":
            summary["disk_modules"].append(module_name)

    summary["devices"] = dict(counter)
    summary["offloaded"] = bool(summary["cpu_modules"] or summary["disk_modules"])
    return summary


def count_parameters_by_device(target_model) -> Dict[str, int]:
    counter: Dict[str, int] = defaultdict(int)
    for parameter in target_model.parameters():
        counter[str(parameter.device)] += parameter.numel()
    return dict(counter)


INPUT_DEVICE = resolve_input_device(model)
DEVICE_MAP_SUMMARY = summarise_device_map(model)
PARAMS_BY_DEVICE = count_parameters_by_device(model)
TOTAL_PARAMS = sum(PARAMS_BY_DEVICE.values())

try:
    FIRST_PARAM_DTYPE = str(next(model.parameters()).dtype)
except StopIteration:
    FIRST_PARAM_DTYPE = "unknown"

QUANTIZATION_CONFIG = getattr(getattr(model, "config", None), "quantization_config", None)

print("=" * 78)
print("MODEL DIAGNOSTICS")
print("=" * 78)
print(f"Model class            : {type(model).__name__}")
print(f"Processor class        : {type(processor).__name__}")
print(f"Tokenizer class        : {type(processor.tokenizer).__name__}")
print(f"Image processor class  : {type(getattr(processor, 'image_processor', None)).__name__}")
print(f"Model dtype (config)   : {getattr(getattr(model, 'config', None), 'dtype', None) or getattr(getattr(model, 'config', None), 'torch_dtype', None)}")
print(f"First parameter dtype  : {FIRST_PARAM_DTYPE}")
print(f"Total parameters       : {TOTAL_PARAMS:,}")
print(f"Model load time        : {MODEL_LOAD_TIME_S}s (processor {PROCESSOR_LOAD_TIME_S}s)")
print(f"Resolved input device  : {INPUT_DEVICE}")
print(f"padding_side           : {processor.tokenizer.padding_side}")
print(f"eos_token_id           : {processor.tokenizer.eos_token_id}")
print(f"pad_token_id           : {processor.tokenizer.pad_token_id}")

print("-" * 78)
print("Quantization configuration")
if QUANTIZATION_CONFIG is None:
    print("  config.quantization_config : absent")
else:
    print(f"  type                       : {type(QUANTIZATION_CONFIG).__name__}")
    try:
        _quant_dict = QUANTIZATION_CONFIG.to_dict()
    except Exception:
        _quant_dict = {
            key: value for key, value in vars(QUANTIZATION_CONFIG).items()
            if not key.startswith("_")
        }
    for _key, _value in sorted(_quant_dict.items()):
        print(f"  {_key:27s}: {_value}")

print("-" * 78)
print("Parameter distribution by device")
for _device, _count in sorted(PARAMS_BY_DEVICE.items()):
    _share = 100.0 * _count / max(TOTAL_PARAMS, 1)
    print(f"  {_device:12s} {_count:>16,} params ({_share:5.1f}%)")

print("-" * 78)
print("Device map")
if DEVICE_MAP_SUMMARY["has_device_map"]:
    for _device, _modules in sorted(DEVICE_MAP_SUMMARY["devices"].items()):
        print(f"  {_device:12s} {_modules} module(s)")
else:
    print("  hf_device_map absent (single-device load)")

print("-" * 78)
print(f"VRAM allocated         : {torch.cuda.memory_allocated() / 1e6:.0f} MB")
print(f"VRAM reserved          : {torch.cuda.memory_reserved() / 1e6:.0f} MB")
print(f"VRAM total             : {torch.cuda.get_device_properties(0).total_memory / 1e6:.0f} MB")
print("=" * 78)

# ------------------------------------------------------------------ warnings --
CPU_OFFLOAD_DETECTED = bool(
    DEVICE_MAP_SUMMARY["offloaded"]
    or any(key.startswith("cpu") or key == "meta" for key in PARAMS_BY_DEVICE)
)

if CPU_OFFLOAD_DETECTED:
    _cpu_params = sum(
        count for device, count in PARAMS_BY_DEVICE.items()
        if device.startswith("cpu") or device == "meta"
    )
    _share = 100.0 * _cpu_params / max(TOTAL_PARAMS, 1)
    print()
    print("!" * 78)
    print("!!  WARNING: CPU / DISK OFFLOAD DETECTED")
    print("!" * 78)
    print(f"!!  {_cpu_params:,} parameters ({_share:.1f}%) are NOT on the GPU.")
    print("!!  Every generated token will cross the PCIe bus. Expect inference to be")
    print("!!  10-50x slower than an on-GPU run. Page timings measured in this state")
    print("!!  are NOT representative and must not be used for projections.")
    print("!!  Modules on CPU/disk (first 10):")
    for _name in (DEVICE_MAP_SUMMARY["cpu_modules"] + DEVICE_MAP_SUMMARY["disk_modules"])[:10]:
        print(f"!!    - {_name}")
    print("!" * 78)
    log("CPU/DISK OFFLOAD DETECTED - performance figures are not representative", logging.WARNING)
else:
    print("\nNo CPU/disk offload detected: the whole model is resident on GPU.")

if INPUT_DEVICE.type != "cuda":
    print("\n[FATAL-ish] Resolved input device is not CUDA. Inference would run on CPU.")
    log(f"Resolved input device is {INPUT_DEVICE} - CPU inference risk", logging.ERROR)

log(
    f"Model diagnostics: class={type(model).__name__} dtype={FIRST_PARAM_DTYPE} "
    f"input_device={INPUT_DEVICE} offload={CPU_OFFLOAD_DETECTED}"
)

## 12 — RAW OCR prompt

One prompt, used for every page of every document type. There is no per-type
prompt, because there is no classification and no field extraction.

The prompt is written in English (the model's strongest instruction language) but
explicitly forbids translating, transliterating or normalising the *content*,
which stays in French / Arabic / English exactly as printed.

Two details that matter in practice:

* **`[UNREADABLE]`** gives the model a legitimate escape hatch. Without one, a VLM
  under pressure to produce text on an illegible region is far more likely to
  hallucinate plausible names and numbers.
* **The MRZ rule is explicit.** `<` filler characters are meaningful data in a
  passport MRZ; a model that "tidies them up" destroys the line. The degeneration
  detector in §14 is built to match this rule, and does not flag legitimate MRZ.

In [ ]:
PROMPT_RAW_OCR = """You are performing literal OCR transcription of a scanned document.

The image is the only source of truth.

Transcribe all readable text visible on the page.

Rules:

- Preserve the original language.
- Do not translate.
- Do not transliterate Arabic.
- Preserve French exactly as visible.
- Preserve English exactly as visible.
- Preserve Arabic exactly as visible.
- Preserve names exactly as visible.
- Preserve numbers exactly.
- Preserve dates exactly.
- Preserve document numbers exactly.
- Preserve account numbers exactly.
- Preserve punctuation when readable.
- Preserve MRZ lines exactly, including every "<" character.
- Include readable handwritten text.
- Keep the reading order and the line breaks of the page.
- Do not infer missing words.
- Do not reconstruct hidden text.
- Do not correct spelling.
- Do not normalize names.
- Do not explain the document.
- Do not summarize the document.
- Do not classify the document.
- Do not output JSON.
- Do not invent values.

When text is genuinely unreadable, write:
[UNREADABLE]

Return only the transcription."""

print("RAW OCR prompt")
print("-" * 78)
print(PROMPT_RAW_OCR)
print("-" * 78)
print(f"Prompt characters: {len(PROMPT_RAW_OCR)}")
print(f"Prompt tokens    : {len(processor.tokenizer(PROMPT_RAW_OCR)['input_ids'])}")

## 13 — Qwen multimodal input and the single inference call

`apply_template()` is copied verbatim from `dom.ipynb`, including the `TypeError`
fallback for implementations that do not accept `enable_thinking`. OCR is a
perception task: chain-of-thought adds latency and gives the model room to
*reason about* the document instead of reading it.

The message structure, the processor invocation (`text=`, `images=`,
`return_tensors="pt"`), the generation flags (`do_sample=False`,
`repetition_penalty=1.0`, `pad_token_id=eos`) and the input-token trimming before
decode are all the proven `dom.ipynb` contract.

### Additions, and why

**1. Stage-level timing.** `template_time_s`, `processor_time_s`,
`device_transfer_time_s`, `generation_time_s` and `decode_time_s` are measured
separately, with `torch.cuda.synchronize()` at the GPU boundaries. Without this
split, "the page took 90 seconds" is unattributable.

**2. Vision-token accounting.** `image_grid_thw` is read back from the processor
output, so the notebook reports how many vision tokens the image actually became
— the number that connects image size to prefill cost.

**3. `stop_reason`.** `EOS` vs `MAX_TOKENS` vs `REPETITION_GUARD`. A page that
stops on `MAX_TOKENS` is either genuinely dense or degenerate, and the two must
not be confused.

**4. A repetition stopping criterion.** When generation collapses into
`!!!!!!!!!!!!`, nothing emits EOS and the model runs to `max_new_tokens` — that
is how one bad page costs 60+ seconds. The criterion halts when the last 96
generated tokens contain ≤ 2 distinct token ids. A passport MRZ cannot trigger
it: its longest `<` run is under 40 characters and it is surrounded by
alphanumerics, so 96 consecutive near-identical tokens never occur. The page is
still recorded as degenerate — it just costs ~3 s instead of ~60 s.

In [ ]:
def apply_template(messages: List[Dict[str, Any]]) -> str:
    """Qwen3: disable 'thinking' when the installed implementation supports it."""
    try:
        return processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
    except TypeError:
        return processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )


def build_messages(prompt: str, image: Image.Image) -> List[Dict[str, Any]]:
    """Exact multimodal message structure proven in dom.ipynb."""
    return [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt},
            ],
        }
    ]


class RepetitionStoppingCriteria(StoppingCriteria):
    """
    Halt a collapsed generation early.

    Fires when the last `window` generated tokens contain at most `max_unique`
    distinct token ids. Deterministic, cheap, and structurally unable to trigger
    on a passport MRZ (whose longest filler run is far below the window).
    """

    def __init__(self, prompt_length: int, window: int = 96, max_unique: int = 2):
        self.prompt_length = int(prompt_length)
        self.window = int(window)
        self.max_unique = int(max_unique)
        self.triggered = False

    def __call__(self, input_ids: torch.LongTensor, scores, **kwargs):
        # A tensor of per-sequence flags is what modern Transformers expects, and
        # older versions accept it too (they only call any() on the result).
        batch_size = int(input_ids.shape[0])
        generated_length = int(input_ids.shape[1]) - self.prompt_length

        done = False
        if generated_length >= self.window:
            tail = input_ids[0, -self.window:]
            if int(torch.unique(tail).numel()) <= self.max_unique:
                self.triggered = True
                done = True

        return torch.full((batch_size,), done, dtype=torch.bool, device=input_ids.device)


def describe_vision_input(inputs: Dict[str, Any]) -> Dict[str, Any]:
    """
    Recover what the processor actually fed to the vision tower.

    `image_grid_thw` is expressed in patches; multiplying by the patch size gives
    the pixel geometry the model really saw, which may differ from the PIL size
    because of MIN_PIXELS / MAX_PIXELS and the 28-pixel alignment.
    """
    info: Dict[str, Any] = {
        "model_image_width": None,
        "model_image_height": None,
        "vision_patches": None,
        "vision_tokens": None,
        "grid_thw": None,
    }

    grid = inputs.get("image_grid_thw", None) if hasattr(inputs, "get") else None
    if grid is None:
        return info

    try:
        values = grid[0].tolist() if hasattr(grid, "tolist") else list(grid[0])
        if len(values) != 3:
            return info
        temporal, height_patches, width_patches = (int(v) for v in values)

        image_processor = getattr(processor, "image_processor", None)
        patch_size = int(getattr(image_processor, "patch_size", 14) or 14)
        merge_size = int(getattr(image_processor, "merge_size", 2) or 2)

        patches = temporal * height_patches * width_patches
        info["grid_thw"] = [temporal, height_patches, width_patches]
        info["vision_patches"] = patches
        info["vision_tokens"] = int(patches // max(merge_size * merge_size, 1))
        info["model_image_height"] = int(height_patches * patch_size)
        info["model_image_width"] = int(width_patches * patch_size)
    except Exception:
        pass

    return info


# Every model call in this notebook goes through qwen_single_call(), so this
# counter is the single authoritative measure of GPU work done in the session.
QWEN_CALL_COUNTER = 0


def qwen_single_call(
    prompt: str,
    image: Image.Image,
    max_new_tokens: int,
) -> Dict[str, Any]:
    """
    ONE Qwen call on ONE image, fully instrumented.

    Returns the decoded text plus every stage timing, token count and stop reason.
    Raises on failure; the caller decides what to do with the exception.
    """
    global QWEN_CALL_COUNTER
    QWEN_CALL_COUNTER += 1

    result: Dict[str, Any] = {
        "text": "",
        "tokens_in": 0,
        "tokens_out": 0,
        "tokens_per_second": 0.0,
        "template_time_s": 0.0,
        "processor_time_s": 0.0,
        "device_transfer_time_s": 0.0,
        "generation_time_s": 0.0,
        "decode_time_s": 0.0,
        "stop_reason": "UNKNOWN",
        "max_new_tokens": int(max_new_tokens),
        "gpu_memory_allocated_before_mb": round(torch.cuda.memory_allocated() / 1e6, 1),
        "gpu_memory_allocated_after_mb": 0.0,
        "gpu_memory_reserved_mb": 0.0,
    }

    messages = build_messages(prompt, image)

    t0 = time.time()
    text_in = apply_template(messages)
    result["template_time_s"] = round(time.time() - t0, 4)

    t0 = time.time()
    inputs = processor(
        text=text_in,
        images=image,
        return_tensors="pt",
    )
    result["processor_time_s"] = round(time.time() - t0, 4)
    result.update(describe_vision_input(inputs))

    t0 = time.time()
    inputs = inputs.to(INPUT_DEVICE)
    torch.cuda.synchronize()
    result["device_transfer_time_s"] = round(time.time() - t0, 4)

    prompt_length = int(inputs["input_ids"].shape[1])
    result["tokens_in"] = prompt_length

    stopping = None
    criteria = None
    if CFG.ENABLE_REPETITION_STOP:
        stopping = RepetitionStoppingCriteria(
            prompt_length=prompt_length,
            window=CFG.REPETITION_STOP_WINDOW,
            max_unique=CFG.REPETITION_STOP_UNIQUE,
        )
        criteria = StoppingCriteriaList([stopping])

    torch.cuda.synchronize()
    t0 = time.time()
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=int(max_new_tokens),
            do_sample=False,
            repetition_penalty=CFG.REPETITION_PENALTY,
            pad_token_id=processor.tokenizer.eos_token_id,
            stopping_criteria=criteria,
        )
    torch.cuda.synchronize()
    result["generation_time_s"] = round(time.time() - t0, 4)

    # Strip the prompt: decode ONLY what the model generated.
    generated_ids = output_ids[0][prompt_length:]
    tokens_out = int(generated_ids.shape[0])
    result["tokens_out"] = tokens_out

    t0 = time.time()
    result["text"] = processor.decode(
        generated_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True,
    )
    result["decode_time_s"] = round(time.time() - t0, 4)

    if stopping is not None and stopping.triggered:
        result["stop_reason"] = "REPETITION_GUARD"
    elif tokens_out >= int(max_new_tokens):
        result["stop_reason"] = "MAX_TOKENS"
    else:
        result["stop_reason"] = "EOS"

    generation_time = max(result["generation_time_s"], 1e-6)
    result["tokens_per_second"] = round(tokens_out / generation_time, 2)
    result["gpu_memory_allocated_after_mb"] = round(torch.cuda.memory_allocated() / 1e6, 1)
    result["gpu_memory_reserved_mb"] = round(torch.cuda.memory_reserved() / 1e6, 1)

    del output_ids, generated_ids, inputs
    return result


print("Inference layer ready:")
print("  apply_template (enable_thinking=False + TypeError fallback)")
print("  build_messages (image + text, dom.ipynb structure)")
print("  qwen_single_call (one call, fully instrumented)")
print(f"  repetition guard: {CFG.ENABLE_REPETITION_STOP} "
      f"(window={CFG.REPETITION_STOP_WINDOW}, max_unique={CFG.REPETITION_STOP_UNIQUE})")

## 14 — Output-degeneration detector

`!!!!!!!!!!!!` is not OCR. It is a hard engineering failure and it must be
detected mechanically, never eyeballed.

### The MRZ trap

A legitimate TD3 passport MRZ looks like this:

```text
P<UTOERIKSSON<<ANNA<MARIA<<<<<<<<<<<<<<<<<<<
L898902C36UTO7408122F1204159ZE184226B<<<<<10
```

That is ~35 % `<` characters and a 20-character filler run. A naive
"too much punctuation" or "long run of one character" rule flags it instantly —
and would throw away exactly the data a KYC pipeline exists to capture.

The detector therefore **segments first**: lines matching the MRZ profile
(`^[A-Z0-9<]{25,50}$`, containing `<`, and at least 35 % alphanumeric) are
identified, counted, and **excluded from the ratio tests**. The tests then run on
the remaining "prose" text. A line of pure `<<<<<<<<<<` fails the alphanumeric
condition, so it is *not* protected as MRZ and is still flagged.

### The eight rules

| Rule | Condition | Catches |
|---|---|---|
| `EMPTY_OUTPUT` | nothing but whitespace | silent generation failure |
| `LOW_UNIQUE_CHARS` | < 6 distinct chars over > 40 chars | `!!!!`, `....`, `aaaa` |
| `LOW_ENTROPY` | Shannon entropy < 2.0 bits/char | any collapsed distribution |
| `CHAR_RUN` | > 30 identical consecutive chars (non-MRZ) | `!!!!!!!!!!!!!!!!!!!!` |
| `PUNCTUATION_FLOOD` | > 55 % punctuation (non-MRZ) | `....,,,,;;;;` |
| `CYCLIC_REPETITION` | a period ≤ 32 repeated ≥ 8 times | `abcabcabc…`, looped phrases |
| `LINE_REPETITION` | ≥ 12 identical consecutive lines | the reported `!!!!` *block* failure |
| `STOP_GUARD` | the repetition stopping criterion fired | collapse caught mid-generation |

**Shannon entropy is the robust core metric.** A raw unique/length ratio is
useless — legitimate 3 000-character French text has ~60 distinct characters,
i.e. a ratio of 0.02, indistinguishable from garbage by ratio alone. Entropy
normalises that away: real multilingual text sits at 4.0–4.6 bits/char, `!!!!`
sits at 0.0, and the gap is unambiguous. The raw ratio is still *recorded*
because the brief asks for it — it is reported, not used as a trigger.

The cell ends with a self-test over pathological strings, a real MRZ block and
genuine French/Arabic text, so the thresholds are demonstrated rather than
asserted.

In [ ]:
PUNCTUATION_CHARS = set("!\"#$%&'()*+,-./:;<=>?@[\\]^_`{|}~«»°…—–·")

MRZ_LINE_RE = re.compile(r"^[A-Z0-9<]{25,50}$")


def is_mrz_line(line: str) -> bool:
    """
    Recognise a machine-readable-zone line.

    TD1/TD2/TD3 lines are 30/36/44 characters of [A-Z0-9<] only. The alphanumeric
    floor is what separates a real MRZ from a line of pure '<' filler, which must
    stay flaggable.
    """
    candidate = line.strip()
    if not MRZ_LINE_RE.match(candidate):
        return False
    if "<" not in candidate:
        return False
    alphanumeric = sum(1 for char in candidate if char.isalnum())
    return alphanumeric / len(candidate) >= 0.35


def shannon_entropy(text: str) -> float:
    """Character-level Shannon entropy in bits per character."""
    if not text:
        return 0.0
    counts = Counter(text)
    total = len(text)
    return float(-sum((count / total) * math.log2(count / total) for count in counts.values()))


def longest_char_run(text: str) -> Tuple[int, Optional[str]]:
    """Length and identity of the longest run of one identical character."""
    if not text:
        return 0, None
    best_length, best_char = 1, text[0]
    current_length, current_char = 1, text[0]
    for char in text[1:]:
        if char == current_char:
            current_length += 1
            if current_length > best_length:
                best_length, best_char = current_length, current_char
        else:
            current_char, current_length = char, 1
    return best_length, best_char


def smallest_period(text: str, max_period: int) -> int:
    """
    Smallest p <= max_period such that text is p-periodic, else len(text).

    Classic test: text has period p iff text[p:] == text[:-p].
    """
    length = len(text)
    if length < 2:
        return length
    for period in range(1, min(int(max_period), length // 2) + 1):
        if text[period:] == text[:-period]:
            return period
    return length


def max_consecutive_line_repeat(text: str) -> Tuple[int, str]:
    """Longest run of identical consecutive non-empty lines."""
    lines = [line.strip() for line in text.splitlines()]
    lines = [line for line in lines if line]
    if not lines:
        return 0, ""
    best_count, best_line = 1, lines[0]
    current_count, current_line = 1, lines[0]
    for line in lines[1:]:
        if line == current_line:
            current_count += 1
            if current_count > best_count:
                best_count, best_line = current_count, current_line
        else:
            current_line, current_count = line, 1
    return best_count, best_line


def detect_degenerate_output(
    text: str,
    stop_reason: str = "EOS",
    cfg: PipelineConfig = None,
) -> Dict[str, Any]:
    """
    Decide whether a transcription is pathological, and report the metrics.

    MRZ lines are excluded from the ratio tests so a valid passport is never
    flagged for its '<' characters.
    """
    cfg = cfg or CFG
    raw = "" if text is None else str(text)

    lines = raw.splitlines()
    mrz_lines = [line for line in lines if is_mrz_line(line)]
    prose_lines = [line for line in lines if not is_mrz_line(line)]
    prose = "\n".join(prose_lines)

    stripped = raw.strip()
    prose_compact = re.sub(r"\s+", "", prose)

    run_length, run_char = longest_char_run(prose_compact)
    line_repeat, repeated_line = max_consecutive_line_repeat(raw)
    unique_chars = set(re.sub(r"\s+", "", stripped))
    punctuation_count = sum(1 for char in prose_compact if char in PUNCTUATION_CHARS)

    period = smallest_period(prose_compact, cfg.DEGEN_PERIOD_MAX) if prose_compact else 0
    period_repeats = (len(prose_compact) // period) if period else 0

    diagnostics: Dict[str, Any] = {
        "text_length": len(raw),
        "text_length_stripped": len(stripped),
        "line_count": len(lines),
        "mrz_line_count": len(mrz_lines),
        "prose_length": len(prose_compact),
        "unique_character_count": len(unique_chars),
        "unique_character_ratio": round(len(unique_chars) / max(len(stripped), 1), 6),
        "shannon_entropy_bits_per_char": round(shannon_entropy(prose_compact or stripped), 4),
        "punctuation_ratio": round(punctuation_count / max(len(prose_compact), 1), 6),
        "longest_repeated_character_run": int(run_length),
        "longest_run_character": run_char,
        "max_consecutive_line_repeat": int(line_repeat),
        "most_repeated_line": repeated_line[:80],
        "smallest_period": int(period),
        "period_repeats": int(period_repeats),
        "stop_reason": stop_reason,
        "degenerate_output": False,
        "degenerate_reason": None,
        "degenerate_reasons": [],
    }

    reasons: List[str] = []

    # 1 - nothing at all
    if not stripped:
        reasons.append("EMPTY_OUTPUT")

    # 8 - the guard already fired mid-generation
    if stop_reason == "REPETITION_GUARD":
        reasons.append("STOP_GUARD")

    long_enough = len(prose_compact) >= cfg.DEGEN_MIN_LENGTH

    # 2 - character vocabulary collapse
    if stripped and long_enough and len(unique_chars) < cfg.DEGEN_MIN_UNIQUE_CHARS:
        reasons.append("LOW_UNIQUE_CHARS")

    # 3 - distribution collapse (the robust core metric)
    if long_enough and diagnostics["shannon_entropy_bits_per_char"] < cfg.DEGEN_ENTROPY_MIN:
        reasons.append("LOW_ENTROPY")

    # 4 - one character hammered
    if run_length > cfg.DEGEN_MAX_CHAR_RUN:
        reasons.append("CHAR_RUN")

    # 5 - punctuation flood (MRZ already removed)
    if long_enough and diagnostics["punctuation_ratio"] > cfg.DEGEN_PUNCT_RATIO_MAX:
        reasons.append("PUNCTUATION_FLOOD")

    # 6 - short cycle repeated many times
    if (
        len(prose_compact) > 64
        and period <= cfg.DEGEN_PERIOD_MAX
        and period_repeats >= cfg.DEGEN_PERIOD_MIN_REPEATS
    ):
        reasons.append("CYCLIC_REPETITION")

    # 7 - the same line over and over
    if line_repeat >= cfg.DEGEN_MAX_LINE_REPEAT:
        reasons.append("LINE_REPETITION")

    diagnostics["degenerate_reasons"] = reasons
    diagnostics["degenerate_output"] = bool(reasons)
    diagnostics["degenerate_reason"] = "+".join(reasons) if reasons else None
    return diagnostics


# ---------------------------------------------------------------- self-test --
_MRZ_SAMPLE = (
    "REPUBLIQUE ALGERIENNE DEMOCRATIQUE ET POPULAIRE\n"
    "PASSEPORT / PASSPORT\n"
    "Nom / Surname: BENALI\n"
    "Prenom / Given names: MOHAMED AMINE\n"
    "P<DZABENALI<<MOHAMED<AMINE<<<<<<<<<<<<<<<<<<<\n"
    "L898902C36DZA7408122M1204159ZE184226B<<<<<10"
)

_FRENCH_SAMPLE = (
    "CONVENTION DE COMPTE DE DEPOT\n"
    "Entre les soussignes, la Banque, ci-apres designee l'etablissement,\n"
    "et le client dont l'identite figure ci-dessous, il a ete convenu ce qui suit.\n"
    "Article 1 - Ouverture du compte : le present compte est ouvert au nom de\n"
    "Monsieur BENALI MOHAMED AMINE, ne le 12/08/1974 a ORAN.\n"
    "Adresse : 15 RUE DES FRERES BOUADOU, ORAN 31000\n"
    "Numero de compte : 00119 2719081 48 DZD\n"
    "Fait a ORAN, le 14/03/2026. Lu et approuve. Signature du client."
)

_ARABIC_SAMPLE = (
    "الجمهورية الجزائرية الديمقراطية الشعبية\n"
    "بطاقة التعريف الوطنية\n"
    "اللقب: بن علي\n"
    "الاسم: محمد أمين\n"
    "تاريخ الميلاد: 1974/08/12"
)

_DEGEN_CASES = [
    ("empty", "", True),
    ("whitespace", "   \n\n  \t ", True),
    ("bangs one line", "!" * 120, True),
    ("bangs block", "\n".join(["!!!!!!!!!!!!"] * 30), True),
    ("dots", "." * 200, True),
    ("angle flood", "<" * 200, True),
    ("cycle", "abc" * 60, True),
    ("repeated line", "\n".join(["Nom : BENALI"] * 25), True),
    ("real MRZ page", _MRZ_SAMPLE, False),
    ("french page", _FRENCH_SAMPLE, False),
    ("arabic page", _ARABIC_SAMPLE, False),
    ("short legit", "FATCA\n[UNREADABLE]\nSignature", False),
]

print("Degeneration detector self-test")
print("=" * 100)
print(f"{'case':20s} {'flag':6s} {'expect':7s} {'entropy':>8s} {'uniq':>5s} {'punct':>6s} {'run':>4s} {'mrz':>4s}  reason")
print("-" * 100)

_degen_failures = 0
for _label, _sample, _expected in _DEGEN_CASES:
    _diag = detect_degenerate_output(_sample)
    _ok = _diag["degenerate_output"] == _expected
    _degen_failures += 0 if _ok else 1
    print(
        f"{_label:20s} "
        f"{str(_diag['degenerate_output']):6s} "
        f"{str(_expected):7s} "
        f"{_diag['shannon_entropy_bits_per_char']:8.3f} "
        f"{_diag['unique_character_count']:5d} "
        f"{_diag['punctuation_ratio']:6.3f} "
        f"{_diag['longest_repeated_character_run']:4d} "
        f"{_diag['mrz_line_count']:4d}  "
        f"{_diag['degenerate_reason']}"
        + ("" if _ok else "   <-- UNEXPECTED")
    )
print("=" * 100)
if _degen_failures:
    raise AssertionError(f"{_degen_failures} degeneration self-test(s) failed - fix thresholds before running OCR.")
print("Detector behaves as specified: pathological output flagged, real MRZ / French / Arabic preserved.")

## 15 — Single-page RAW OCR

The heart of the pipeline, and deliberately the *only* place a model call is made.

```text
render (standard)
  -> blank check            -> BLANK_PAGE, no model call
  -> conservative preprocess
  -> ONE Qwen call
  -> degeneration check
  -> [optional] ONE HD fallback call, only if validation failed
```

### Fallback policy

`dom.ipynb` escalated through up to five crops per page because it was hunting
for *fields*. RAW OCR has no fields to chase, so escalation is a repair
mechanism, not a strategy. A fallback fires only when the first pass is
demonstrably bad:

| Trigger | Meaning |
|---|---|
| `DEGENERATE` | the detector flagged the output |
| `EMPTY_OUTPUT` | the model returned nothing |
| `SUSPICIOUSLY_SHORT` | ≥ 2 % of the page is ink, yet < 60 characters came back |
| `GENERATION_ERROR` | the first call raised |

The fallback is exactly **one** additional call at HD resolution. If it also
fails, the page is recorded as failed — there is no third attempt. The
budget is therefore 1 call per page nominally, 2 in the worst case, and
`fallback_used` is recorded so the extra cost is always visible.

### Statuses

`SUCCESS` · `SUCCESS_FALLBACK` · `BLANK_PAGE` · `DEGENERATE` · `FAILED`

In [ ]:
def evaluate_transcription(
    text: str,
    stop_reason: str,
    page_ink_ratio: float,
    cfg: PipelineConfig = None,
) -> Tuple[Dict[str, Any], List[str]]:
    """
    Validate one transcription.

    Returns (degeneration diagnostics, list of fallback triggers).
    """
    cfg = cfg or CFG
    diagnostics = detect_degenerate_output(text, stop_reason=stop_reason, cfg=cfg)
    triggers: List[str] = []

    if diagnostics["degenerate_output"]:
        triggers.append(f"DEGENERATE:{diagnostics['degenerate_reason']}")

    stripped = (text or "").strip()
    if not stripped:
        if "DEGENERATE:EMPTY_OUTPUT" not in triggers:
            triggers.append("EMPTY_OUTPUT")
    elif page_ink_ratio >= cfg.FALLBACK_MIN_INK_RATIO and len(stripped) < cfg.FALLBACK_MIN_CHARS:
        triggers.append("SUSPICIOUSLY_SHORT")

    return diagnostics, triggers


def print_qwen_start(context: Dict[str, Any], image: Image.Image, pass_label: str) -> None:
    print(f"\n[QWEN START] {pass_label}")
    print(f"Customer     : {context.get('customer_id')}")
    print(f"Document     : {context.get('logical_document_type')}")
    print(f"Physical file: {context.get('physical_filename')}")
    print(f"Page         : {context.get('page_number')}/{context.get('page_count')}")
    print(f"Image        : {image.width}x{image.height}")


def print_qwen_end(call: Dict[str, Any], status: str, fallback_used: bool, degenerate: bool) -> None:
    print("[QWEN END]")
    print(f"Generation   : {call['generation_time_s']:.2f} s")
    print(f"Input tokens : {call['tokens_in']}")
    print(f"Output tokens: {call['tokens_out']}")
    print(f"Tokens/sec   : {call['tokens_per_second']:.2f}")
    print(f"Vision tokens: {call.get('vision_tokens')}")
    print(f"Stop reason  : {call['stop_reason']}")
    print(f"Status       : {status}")
    print(f"Degenerate   : {'YES' if degenerate else 'NO'}")
    print(f"Fallback     : {'YES' if fallback_used else 'NO'}")


def process_page(
    pdf_path: Path,
    page_index: int,
    context: Dict[str, Any],
    cfg: PipelineConfig = None,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    RAW OCR of one PDF page: render -> preprocess -> one Qwen call -> validate.

    Never raises: every failure is captured in status / failure_type /
    failure_message so that one bad page cannot stop the pipeline.
    """
    cfg = cfg or CFG
    page_started = time.time()

    record: Dict[str, Any] = {
        "pipeline_version": PIPELINE_VERSION,
        "customer_id": context.get("customer_id"),
        "logical_document_type": context.get("logical_document_type"),
        "physical_filename": context.get("physical_filename"),
        "pdf_sha256": context.get("pdf_sha256"),
        "pdf_path": str(pdf_path),
        "page_index": int(page_index),
        "page_number": int(page_index) + 1,
        "page_count": context.get("page_count"),
        "raw_ocr": "",

        "render_time_s": 0.0,
        "preprocess_time_s": 0.0,
        "template_time_s": 0.0,
        "processor_time_s": 0.0,
        "device_transfer_time_s": 0.0,
        "generation_time_s": 0.0,
        "decode_time_s": 0.0,
        "validation_time_s": 0.0,
        "total_time_s": 0.0,

        "render_width": None,
        "render_height": None,
        "processed_width": None,
        "processed_height": None,
        "model_image_width": None,
        "model_image_height": None,
        "vision_tokens": None,
        "white_ratio": None,
        "ink_ratio": None,
        "page_rotation": None,

        "tokens_in": 0,
        "tokens_out": 0,
        "tokens_per_second": 0.0,
        "stop_reason": None,
        "max_new_tokens": None,

        "gpu_memory_allocated_before_mb": round(torch.cuda.memory_allocated() / 1e6, 1),
        "gpu_memory_allocated_after_mb": None,
        "gpu_memory_reserved_mb": None,

        "status": "PENDING",
        "failure_type": None,
        "failure_message": None,
        "fallback_used": False,
        "fallback_triggers": [],
        "qwen_calls": 0,
        "degenerate": False,
        "degenerate_reason": None,
        "diagnostics": {},
        "preprocessing": {},
        "processed_at": datetime.now().isoformat(timespec="seconds"),
    }

    # ------------------------------------------------------------- render ----
    try:
        rendered = render_pdf_page(
            pdf_path,
            page_index,
            zoom=cfg.RENDER_ZOOM_STANDARD,
            max_side=cfg.IMAGE_MAX_SIZE_STANDARD,
        )
    except Exception as exc:
        record["status"] = "FAILED"
        record["failure_type"] = "RENDER_ERROR"
        record["failure_message"] = repr(exc)
        record["total_time_s"] = round(time.time() - page_started, 3)
        return record

    image = rendered["image"]
    record.update({
        "render_time_s": rendered["render_time_s"],
        "render_width": rendered["render_width"],
        "render_height": rendered["render_height"],
        "processed_width": rendered["processed_width"],
        "processed_height": rendered["processed_height"],
        "page_count": rendered["page_count"],
        "white_ratio": rendered["white_ratio"],
        "page_rotation": rendered["page_rotation"],
    })

    page_ink = ink_ratio(image)
    record["ink_ratio"] = round(page_ink, 6)

    # -------------------------------------------------------- blank page ----
    if cfg.SKIP_BLANK_PAGES and rendered["white_ratio"] >= cfg.BLANK_THRESHOLD:
        record["status"] = "BLANK_PAGE"
        record["total_time_s"] = round(time.time() - page_started, 3)
        if verbose:
            print(
                f"\n[BLANK] {context.get('customer_id')} | {context.get('logical_document_type')} | "
                f"page {record['page_number']}/{record['page_count']} | "
                f"white_ratio={rendered['white_ratio']:.4f} >= {cfg.BLANK_THRESHOLD} | no Qwen call"
            )
        return record

    # ---------------------------------------------------------- preprocess --
    try:
        image, preprocessing = preprocess_image(image, cfg)
        record["preprocessing"] = preprocessing
        record["preprocess_time_s"] = preprocessing.get("preprocess_time_s", 0.0)
        record["processed_width"] = image.width
        record["processed_height"] = image.height
    except Exception as exc:
        # Preprocessing must never cost us a page: fall back to the raw render.
        record["preprocessing"] = {"error": repr(exc)}
        log(f"Preprocessing failed, using raw render: {exc!r}", logging.WARNING)

    # ------------------------------------------------------ pass 1 (std) ----
    context_for_print = dict(context)
    context_for_print["page_number"] = record["page_number"]
    context_for_print["page_count"] = record["page_count"]

    if verbose:
        print_qwen_start(context_for_print, image, "pass 1 / standard")

    call: Optional[Dict[str, Any]] = None
    try:
        call = qwen_single_call(PROMPT_RAW_OCR, image, cfg.MAX_NEW_TOKENS_RAW_OCR)
        record["qwen_calls"] += 1
    except torch.cuda.OutOfMemoryError as exc:
        record["failure_type"] = "CUDA_OOM"
        record["failure_message"] = repr(exc)
        recover_from_oom()
    except Exception as exc:
        record["failure_type"] = "GENERATION_ERROR"
        record["failure_message"] = repr(exc)

    triggers: List[str] = []
    diagnostics: Dict[str, Any] = {}

    if call is not None:
        t_validation = time.time()
        try:
            diagnostics, triggers = evaluate_transcription(
                call["text"], call["stop_reason"], page_ink, cfg
            )
        except Exception as exc:
            # Validation must never cost a transcription we already paid for.
            diagnostics = {"degenerate_output": False, "degenerate_reason": None,
                           "validation_error": repr(exc)}
            triggers = []
            log(f"Validation failed (transcription kept): {exc!r}", logging.WARNING)
        record["validation_time_s"] = round(time.time() - t_validation, 4)
        _absorb_call(record, call, diagnostics)
        if verbose:
            print_qwen_end(
                call,
                status="OK" if not triggers else "NEEDS_FALLBACK",
                fallback_used=False,
                degenerate=diagnostics["degenerate_output"],
            )
    else:
        triggers = [record["failure_type"] or "GENERATION_ERROR"]

    record["fallback_triggers"] = triggers

    # ---------------------------------------------------- pass 2 (HD) -------
    if triggers and cfg.ENABLE_HD_FALLBACK:
        if verbose:
            print(f"[FALLBACK] triggers = {triggers} -> one HD retry")
        try:
            hd_rendered = render_pdf_page(
                pdf_path,
                page_index,
                zoom=cfg.RENDER_ZOOM_HD,
                max_side=cfg.IMAGE_MAX_SIZE_HD,
            )
            hd_image = hd_rendered["image"]
            record["render_time_s"] = round(
                record["render_time_s"] + hd_rendered["render_time_s"], 4
            )

            if cfg.ENABLE_PREPROCESSING:
                hd_image, hd_preprocessing = preprocess_image(hd_image, cfg)
                record["preprocessing"] = {
                    "standard": record.get("preprocessing"),
                    "hd": hd_preprocessing,
                }
                record["preprocess_time_s"] = round(
                    record["preprocess_time_s"] + hd_preprocessing.get("preprocess_time_s", 0.0), 4
                )

            if verbose:
                print_qwen_start(context_for_print, hd_image, "pass 2 / HD fallback")

            hd_call = qwen_single_call(PROMPT_RAW_OCR, hd_image, cfg.MAX_NEW_TOKENS_RAW_OCR_HD)
            record["qwen_calls"] += 1
            record["fallback_used"] = True

            t_validation = time.time()
            try:
                hd_diagnostics, hd_triggers = evaluate_transcription(
                    hd_call["text"], hd_call["stop_reason"], ink_ratio(hd_image), cfg
                )
            except Exception as exc:
                hd_diagnostics = {"degenerate_output": False, "degenerate_reason": None,
                                  "validation_error": repr(exc)}
                hd_triggers = []
                log(f"HD validation failed (transcription kept): {exc!r}", logging.WARNING)
            record["validation_time_s"] = round(
                record["validation_time_s"] + (time.time() - t_validation), 4
            )

            # Keep the HD result when it is not worse than pass 1.
            keep_hd = (not hd_triggers) or (call is None) or (
                len((hd_call["text"] or "").strip()) > len((call["text"] or "").strip())
            )
            if keep_hd:
                _absorb_call(record, hd_call, hd_diagnostics, accumulate_timings=True)
                triggers = hd_triggers
                diagnostics = hd_diagnostics
            else:
                _accumulate_timings_only(record, hd_call)

            record["fallback_triggers"] = record["fallback_triggers"] + [
                f"AFTER_HD:{t}" for t in hd_triggers
            ]

            if verbose:
                print_qwen_end(
                    hd_call,
                    status="OK" if not hd_triggers else "STILL_BAD",
                    fallback_used=True,
                    degenerate=hd_diagnostics["degenerate_output"],
                )

        except torch.cuda.OutOfMemoryError as exc:
            record["failure_type"] = "CUDA_OOM_FALLBACK"
            record["failure_message"] = repr(exc)
            recover_from_oom()
        except Exception as exc:
            record["failure_type"] = record["failure_type"] or "FALLBACK_ERROR"
            record["failure_message"] = record["failure_message"] or repr(exc)

    # ------------------------------------------------------------ status ----
    if record["qwen_calls"] == 0:
        record["status"] = "FAILED"
        record["failure_type"] = record["failure_type"] or "NO_MODEL_CALL"
    elif diagnostics.get("degenerate_output"):
        record["status"] = "DEGENERATE"
        record["failure_type"] = record["failure_type"] or "DEGENERATE_OUTPUT"
    elif not (record["raw_ocr"] or "").strip():
        record["status"] = "FAILED"
        record["failure_type"] = record["failure_type"] or "EMPTY_TRANSCRIPTION"
    elif record["fallback_used"]:
        record["status"] = "SUCCESS_FALLBACK"
    else:
        record["status"] = "SUCCESS"

    # Aggregate throughput over every call actually made on this page.
    if record["generation_time_s"] > 0:
        record["tokens_per_second"] = round(record["tokens_out"] / record["generation_time_s"], 2)

    record["total_time_s"] = round(time.time() - page_started, 3)
    record["gpu_memory_allocated_after_mb"] = round(torch.cuda.memory_allocated() / 1e6, 1)
    record["gpu_memory_reserved_mb"] = round(torch.cuda.memory_reserved() / 1e6, 1)

    if record["total_time_s"] > cfg.SLOW_PAGE_WARNING_S:
        print(
            f"\n{'*' * 78}\n"
            f"*  SLOW PAGE WARNING: {record['total_time_s']:.1f}s > {cfg.SLOW_PAGE_WARNING_S}s\n"
            f"*  generation={record['generation_time_s']:.1f}s  tokens_out={record['tokens_out']}  "
            f"tokens/s={record['tokens_per_second']:.1f}  stop={record['stop_reason']}\n"
            f"*  vision_tokens={record['vision_tokens']}  image={record['processed_width']}x{record['processed_height']}\n"
            f"*  Check section 23 for the stage breakdown before launching a large run.\n"
            f"{'*' * 78}"
        )

    return record


def _absorb_call(
    record: Dict[str, Any],
    call: Dict[str, Any],
    diagnostics: Dict[str, Any],
    accumulate_timings: bool = False,
) -> None:
    """Copy a model call's output and metrics into the page record."""
    record["raw_ocr"] = call["text"]
    record["tokens_in"] = (record["tokens_in"] + call["tokens_in"]) if accumulate_timings else call["tokens_in"]
    record["tokens_out"] = (record["tokens_out"] + call["tokens_out"]) if accumulate_timings else call["tokens_out"]
    record["tokens_per_second"] = call["tokens_per_second"]
    record["stop_reason"] = call["stop_reason"]
    record["max_new_tokens"] = call["max_new_tokens"]
    record["model_image_width"] = call.get("model_image_width")
    record["model_image_height"] = call.get("model_image_height")
    record["vision_tokens"] = call.get("vision_tokens")

    for key in ("template_time_s", "processor_time_s", "device_transfer_time_s",
                "generation_time_s", "decode_time_s"):
        record[key] = round(record[key] + call[key], 4) if accumulate_timings else call[key]

    record["degenerate"] = bool(diagnostics.get("degenerate_output"))
    record["degenerate_reason"] = diagnostics.get("degenerate_reason")
    record["diagnostics"] = diagnostics


def _accumulate_timings_only(record: Dict[str, Any], call: Dict[str, Any]) -> None:
    """A discarded fallback call still costs time and tokens: account for it."""
    record["tokens_in"] += call["tokens_in"]
    record["tokens_out"] += call["tokens_out"]
    for key in ("template_time_s", "processor_time_s", "device_transfer_time_s",
                "generation_time_s", "decode_time_s"):
        record[key] = round(record[key] + call[key], 4)


def recover_from_oom() -> None:
    """
    CUDA OOM recovery: record, release, continue only if the context survives.

    empty_cache() is NOT called after every page (it costs time); it is called
    here and at document boundaries, which is where it actually helps.
    """
    log("CUDA OOM caught - releasing cached memory", logging.ERROR)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.synchronize()
            # Touch the device: if the context is dead this raises instead of
            # letting the pipeline continue against a broken GPU.
            float(torch.zeros(8, device="cuda").sum().item())
            log("CUDA context still usable after OOM")
        except Exception as exc:
            log(f"CUDA context unusable after OOM: {exc!r}", logging.ERROR)
            raise RuntimeError("CUDA context is unusable after OOM; restart the kernel.") from exc


print("Single-page RAW OCR ready: process_page()")
print(f"  pass 1 : zoom={CFG.RENDER_ZOOM_STANDARD} max_side={CFG.IMAGE_MAX_SIZE_STANDARD} "
      f"max_new_tokens={CFG.MAX_NEW_TOKENS_RAW_OCR}")
print(f"  pass 2 : zoom={CFG.RENDER_ZOOM_HD} max_side={CFG.IMAGE_MAX_SIZE_HD} "
      f"max_new_tokens={CFG.MAX_NEW_TOKENS_RAW_OCR_HD} (enabled={CFG.ENABLE_HD_FALLBACK})")

## 16 — Single-page benchmark (the performance gate)

**Do not run 50 pages yet.** This cell processes exactly **one** representative
page, then the **same** page a second time, and compares cold against warm.

Why the repeat matters: the first call in a process pays for CUDA kernel
autotuning, cuDNN algorithm selection, lazy module initialisation and allocator
warm-up. A cold page can easily be 2–4× a warm one. Projecting a dataset runtime
from the cold number is how a 20-minute job gets estimated at two hours — or the
reverse.

The cell then prints an empirical projection for 10 / 50 / 100 pages from the
**warm** figure, and raises a prominent warning if the page is abnormally slow or
throughput is abnormally low.

This is the gate: **read the numbers here before running anything larger.**

In [ ]:
def pick_benchmark_page(inventory: pd.DataFrame) -> Optional[Dict[str, Any]]:
    """
    Pick one representative page, reproducibly.

    Preference order: an identity document (densest and hardest of the five),
    otherwise the first available target document of the first customer.
    """
    available = inventory[inventory["exists"]].copy()
    if available.empty:
        return None

    preference = {name: rank for rank, name in enumerate(TARGET_DOCUMENTS)}
    available["_rank"] = available["expected_document_type"].map(preference).fillna(99)
    available = available.sort_values(["customer_id", "_rank"], kind="mergesort")

    row = available.iloc[0]
    pdf_path = Path(row["full_path"])
    try:
        page_count = pdf_page_count(pdf_path)
    except Exception as exc:
        log(f"Benchmark candidate unreadable ({pdf_path.name}): {exc!r}", logging.WARNING)
        return None

    return {
        "customer_id": str(row["customer_id"]),
        "logical_document_type": str(row["expected_document_type"]),
        "physical_filename": str(row["matched_filename"]),
        "pdf_path": pdf_path,
        "page_count": page_count,
        "page_index": 0,
    }


def format_duration(seconds: float) -> str:
    """HH:MM:SS / MM:SS (same helper shape as dom.ipynb)."""
    seconds = max(0, int(round(float(seconds or 0))))
    hours, remainder = divmod(seconds, 3600)
    minutes, secs = divmod(remainder, 60)
    if hours:
        return f"{hours:02d}:{minutes:02d}:{secs:02d}"
    return f"{minutes:02d}:{secs:02d}"


def print_page_metrics(title: str, record: Dict[str, Any]) -> None:
    print(f"\n--- {title} ---")
    print(f"  render time        : {record['render_time_s']:.3f} s")
    print(f"  preprocess time    : {record['preprocess_time_s']:.3f} s")
    print(f"  template time      : {record['template_time_s']:.3f} s")
    print(f"  processor time     : {record['processor_time_s']:.3f} s")
    print(f"  device transfer    : {record['device_transfer_time_s']:.3f} s")
    print(f"  generation time    : {record['generation_time_s']:.3f} s")
    print(f"  decode time        : {record['decode_time_s']:.3f} s")
    print(f"  validation time    : {record['validation_time_s']:.3f} s")
    print(f"  TOTAL              : {record['total_time_s']:.3f} s")
    print(f"  tokens in          : {record['tokens_in']}")
    print(f"  tokens out         : {record['tokens_out']}")
    print(f"  tokens/sec         : {record['tokens_per_second']:.2f}")
    print(f"  vision tokens      : {record['vision_tokens']}")
    print(f"  image (PIL)        : {record['processed_width']}x{record['processed_height']}")
    print(f"  image (model)      : {record['model_image_width']}x{record['model_image_height']}")
    print(f"  stop reason        : {record['stop_reason']}")
    print(f"  status             : {record['status']}")
    print(f"  degenerate         : {record['degenerate']} ({record['degenerate_reason']})")
    print(f"  fallback used      : {record['fallback_used']}")
    print(f"  GPU alloc before/after: {record['gpu_memory_allocated_before_mb']:.0f} / "
          f"{record['gpu_memory_allocated_after_mb']:.0f} MB")
    print(f"  GPU reserved       : {record['gpu_memory_reserved_mb']:.0f} MB")


BENCHMARK_TARGET = pick_benchmark_page(INVENTORY_DF)

if BENCHMARK_TARGET is None:
    raise RuntimeError("No target PDF available for the benchmark. Check the inventory (section 6).")

print("=" * 78)
print("SINGLE-PAGE BENCHMARK")
print("=" * 78)
print(f"Customer      : {BENCHMARK_TARGET['customer_id']}")
print(f"Document      : {BENCHMARK_TARGET['logical_document_type']}")
print(f"Physical file : {BENCHMARK_TARGET['physical_filename']}")
print(f"Pages in PDF  : {BENCHMARK_TARGET['page_count']}")
print("Benchmark page: 1")

_bench_context = {
    "customer_id": BENCHMARK_TARGET["customer_id"],
    "logical_document_type": BENCHMARK_TARGET["logical_document_type"],
    "physical_filename": BENCHMARK_TARGET["physical_filename"],
    "pdf_sha256": None,  # not needed for a benchmark, and hashing is timed separately
    "page_count": BENCHMARK_TARGET["page_count"],
}

BENCH_COLD = process_page(
    BENCHMARK_TARGET["pdf_path"], BENCHMARK_TARGET["page_index"], _bench_context, CFG, verbose=True
)
print_page_metrics("COLD (first call in this process)", BENCH_COLD)

BENCH_WARM = process_page(
    BENCHMARK_TARGET["pdf_path"], BENCHMARK_TARGET["page_index"], _bench_context, CFG, verbose=True
)
print_page_metrics("WARM (same page, kernels and allocator warmed up)", BENCH_WARM)

# ----------------------------------------------------------- comparison -----
print("\n" + "=" * 78)
print("COLD vs WARM")
print("=" * 78)
print(f"{'stage':22s} {'cold (s)':>10s} {'warm (s)':>10s} {'delta':>10s}")
print("-" * 78)
for _stage in ("render_time_s", "preprocess_time_s", "template_time_s", "processor_time_s",
               "device_transfer_time_s", "generation_time_s", "decode_time_s",
               "validation_time_s", "total_time_s"):
    _cold, _warm = float(BENCH_COLD[_stage]), float(BENCH_WARM[_stage])
    print(f"{_stage:22s} {_cold:10.3f} {_warm:10.3f} {(_warm - _cold):+10.3f}")
print("-" * 78)
print(f"{'tokens out':22s} {BENCH_COLD['tokens_out']:10d} {BENCH_WARM['tokens_out']:10d}")
print(f"{'tokens/sec':22s} {BENCH_COLD['tokens_per_second']:10.2f} {BENCH_WARM['tokens_per_second']:10.2f}")

_warm_total = max(float(BENCH_WARM["total_time_s"]), 1e-6)
_speedup = float(BENCH_COLD["total_time_s"]) / _warm_total

print("=" * 78)
print(f"Cold/warm speedup   : {_speedup:.2f}x")
print(f"Qwen calls used so far (benchmark included): {QWEN_CALL_COUNTER}")
print()
print("Empirical projections, based on the WARM page:")
for _pages in (10, 50, 100):
    print(f"  {_pages:4d} pages -> {format_duration(_warm_total * _pages)}  "
          f"({_warm_total * _pages:.0f} s)")
print("=" * 78)

# ------------------------------------------------------------- warnings -----
_warnings_raised = []
if _warm_total > CFG.SLOW_PAGE_WARNING_S:
    _warnings_raised.append(
        f"Warm page takes {_warm_total:.1f}s (> {CFG.SLOW_PAGE_WARNING_S}s). "
        f"50 pages would take {format_duration(_warm_total * 50)}."
    )
if BENCH_WARM["tokens_per_second"] and BENCH_WARM["tokens_per_second"] < CFG.LOW_THROUGHPUT_WARNING_TPS:
    _warnings_raised.append(
        f"Throughput is {BENCH_WARM['tokens_per_second']:.1f} tok/s "
        f"(< {CFG.LOW_THROUGHPUT_WARNING_TPS}). Expected 25-40 tok/s for a 27B bf16 model on an H100. "
        "Check section 11 for CPU offload."
    )
if BENCH_WARM["stop_reason"] == "MAX_TOKENS":
    _warnings_raised.append(
        f"Generation hit max_new_tokens ({BENCH_WARM['max_new_tokens']}). The page is either "
        "genuinely very dense or the output is degenerate. Inspect the transcription before raising the limit."
    )
if BENCH_WARM["degenerate"]:
    _warnings_raised.append(f"Benchmark page is DEGENERATE: {BENCH_WARM['degenerate_reason']}")
if CPU_OFFLOAD_DETECTED:
    _warnings_raised.append("CPU/disk offload is active: these timings are not representative.")

if _warnings_raised:
    print()
    print("!" * 78)
    print("!!  BENCHMARK WARNINGS")
    print("!" * 78)
    for _message in _warnings_raised:
        print(f"!!  - {_message}")
    print("!" * 78)
    for _message in _warnings_raised:
        log(f"BENCHMARK WARNING: {_message}", logging.WARNING)
else:
    print("\nBenchmark looks healthy. Proceed to the diagnostic run.")

print("\nFirst 1200 characters of the WARM transcription:")
print("-" * 78)
print((BENCH_WARM["raw_ocr"] or "")[:1200])
print("-" * 78)

## 17 — Single-PDF processor

One PDF → its SHA-256 → every page → one page record each.

* the page count is read dynamically (`doc.page_count`), never assumed;
* each page is checkpointed **as soon as it completes**, so a kernel interrupt
  costs at most one page of work;
* a failing page is recorded and the loop continues;
* `gc.collect()` + `torch.cuda.empty_cache()` run **once per document**, not once
  per page — `empty_cache()` forces the allocator to give memory back to the
  driver and re-acquiring it costs time, so calling it on every page is a net
  loss. Section 15 calls it additionally on an OOM, which is where it earns its
  keep.

In [ ]:
def process_pdf(
    pdf_path: Path,
    customer_id: str,
    logical_document_type: str,
    cfg: PipelineConfig = None,
    verbose: bool = True,
    call_budget: Optional[Dict[str, int]] = None,
) -> Dict[str, Any]:
    """
    RAW OCR of every page of one PDF.

    `call_budget` is a mutable {"remaining": n} dict; when it reaches zero the
    remaining pages are recorded as SKIPPED_BUDGET instead of being processed.
    Never raises: PDF-level failures are captured in the returned summary.
    """
    cfg = cfg or CFG
    pdf_path = Path(pdf_path)
    started = time.time()

    summary: Dict[str, Any] = {
        "customer_id": customer_id,
        "logical_document_type": logical_document_type,
        "physical_filename": pdf_path.name,
        "pdf_path": str(pdf_path),
        "pdf_sha256": None,
        "page_count": 0,
        "pages_processed": 0,
        "pages_reused": 0,
        "pages_skipped_blank": 0,
        "pages_skipped_budget": 0,
        "pages_failed": 0,
        "pages_degenerate": 0,
        "records": [],
        "status": "OK",
        "failure_type": None,
        "failure_message": None,
        "elapsed_s": 0.0,
    }

    try:
        summary["pdf_sha256"] = sha256_file(pdf_path)
        summary["page_count"] = pdf_page_count(pdf_path)
    except Exception as exc:
        summary["status"] = "FAILED"
        summary["failure_type"] = "PDF_OPEN_ERROR"
        summary["failure_message"] = repr(exc)
        summary["elapsed_s"] = round(time.time() - started, 3)
        log(f"PDF unreadable: {pdf_path.name} -> {exc!r}", logging.ERROR)
        return summary

    context = {
        "customer_id": customer_id,
        "logical_document_type": logical_document_type,
        "physical_filename": pdf_path.name,
        "pdf_sha256": summary["pdf_sha256"],
        "page_count": summary["page_count"],
    }

    if verbose:
        print(f"\n{'=' * 78}")
        print(f"PDF  : {pdf_path.name}")
        print(f"Type : {logical_document_type}   Customer: {customer_id}")
        print(f"Pages: {summary['page_count']}   SHA-256: {summary['pdf_sha256'][:16]}...")
        print("=" * 78)

    for page_index in range(summary["page_count"]):
        # ---- resume from checkpoint ----------------------------------------
        existing = load_page_checkpoint(context, page_index + 1, cfg)
        if existing is not None:
            summary["records"].append(existing)
            summary["pages_reused"] += 1
            if verbose:
                print(
                    f"[REUSE] page {page_index + 1}/{summary['page_count']} "
                    f"| status={existing.get('status')} | checkpoint hit"
                )
            continue

        # ---- call budget ----------------------------------------------------
        if call_budget is not None and call_budget.get("remaining", 0) <= 0:
            summary["records"].append({
                "pipeline_version": PIPELINE_VERSION,
                "customer_id": customer_id,
                "logical_document_type": logical_document_type,
                "physical_filename": pdf_path.name,
                "pdf_sha256": summary["pdf_sha256"],
                "pdf_path": str(pdf_path),
                "page_index": page_index,
                "page_number": page_index + 1,
                "page_count": summary["page_count"],
                "raw_ocr": "",
                "status": "SKIPPED_BUDGET",
                "failure_type": "MAX_QWEN_CALLS_REACHED",
                "failure_message": "Qwen call budget exhausted in diagnostic mode",
                "total_time_s": 0.0,
                "tokens_in": 0,
                "tokens_out": 0,
                "degenerate": False,
                "fallback_used": False,
                "qwen_calls": 0,
                "processed_at": datetime.now().isoformat(timespec="seconds"),
            })
            summary["pages_skipped_budget"] += 1
            if verbose:
                print(f"[BUDGET] page {page_index + 1}/{summary['page_count']} skipped (MAX_QWEN_CALLS reached)")
            continue

        # ---- process ---------------------------------------------------------
        try:
            record = process_page(pdf_path, page_index, context, cfg, verbose=verbose)
        except Exception as exc:
            # process_page is defensive, but never let one page kill the PDF.
            record = {
                "pipeline_version": PIPELINE_VERSION,
                "customer_id": customer_id,
                "logical_document_type": logical_document_type,
                "physical_filename": pdf_path.name,
                "pdf_sha256": summary["pdf_sha256"],
                "pdf_path": str(pdf_path),
                "page_index": page_index,
                "page_number": page_index + 1,
                "page_count": summary["page_count"],
                "raw_ocr": "",
                "status": "FAILED",
                "failure_type": "UNHANDLED_PAGE_ERROR",
                "failure_message": repr(exc),
                "total_time_s": 0.0,
                "tokens_in": 0,
                "tokens_out": 0,
                "degenerate": False,
                "fallback_used": False,
                "qwen_calls": 0,
                "processed_at": datetime.now().isoformat(timespec="seconds"),
            }
            log(f"Unhandled page error {pdf_path.name} p{page_index + 1}: {exc!r}", logging.ERROR)

        if call_budget is not None:
            call_budget["remaining"] = call_budget.get("remaining", 0) - int(record.get("qwen_calls", 0) or 0)

        # ---- persist immediately --------------------------------------------
        try:
            save_page_checkpoint(record, cfg)
        except Exception as exc:
            log(f"Checkpoint write failed for {pdf_path.name} p{page_index + 1}: {exc!r}", logging.ERROR)
            record["failure_message"] = (record.get("failure_message") or "") + f" | CHECKPOINT_WRITE_FAILED {exc!r}"

        summary["records"].append(record)

        status = record.get("status")
        if status == "BLANK_PAGE":
            summary["pages_skipped_blank"] += 1
        elif status in ("SUCCESS", "SUCCESS_FALLBACK"):
            summary["pages_processed"] += 1
        elif status == "DEGENERATE":
            summary["pages_degenerate"] += 1
            summary["pages_processed"] += 1
        else:
            summary["pages_failed"] += 1

    # ---- document-level cleanup ---------------------------------------------
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    summary["elapsed_s"] = round(time.time() - started, 3)

    if verbose:
        print(
            f"\n[PDF DONE] {pdf_path.name} | pages={summary['page_count']} "
            f"| ok={summary['pages_processed']} "
            f"| reused={summary['pages_reused']} "
            f"| blank={summary['pages_skipped_blank']} "
            f"| degenerate={summary['pages_degenerate']} "
            f"| failed={summary['pages_failed']} "
            f"| budget_skipped={summary['pages_skipped_budget']} "
            f"| {summary['elapsed_s']:.1f}s"
        )

    log(
        f"PDF processed: {customer_id}/{pdf_path.name} pages={summary['page_count']} "
        f"ok={summary['pages_processed']} failed={summary['pages_failed']} "
        f"degenerate={summary['pages_degenerate']} elapsed={summary['elapsed_s']}s"
    )
    return summary


print("Single-PDF processor ready: process_pdf()")

## 18 — Single-customer processor

One customer → their available target documents → every physical file.

Duplicates: with `CFG.PROCESS_DUPLICATES = True` (the default for a RAW
diagnostic run) **every** physical file that maps to a logical type is processed,
not just the first. Checkpoint keys include the physical filename, so
`FATCA.PDF` and `FATCA (1).PDF` never collide.

In [ ]:
def customer_documents(inventory: pd.DataFrame, customer_id: str, cfg: PipelineConfig = None) -> List[Dict[str, Any]]:
    """
    Physical files to process for one customer, in canonical document order.

    Honours CFG.PROCESS_DUPLICATES: all matching files, or only the primary one.
    """
    cfg = cfg or CFG
    rows = inventory[(inventory["customer_id"] == customer_id) & (inventory["exists"])]
    documents: List[Dict[str, Any]] = []

    order = {name: rank for rank, name in enumerate(TARGET_DOCUMENTS)}
    rows = rows.assign(_rank=rows["expected_document_type"].map(order).fillna(99)).sort_values(
        "_rank", kind="mergesort"
    )

    for _, row in rows.iterrows():
        paths = [item for item in str(row["duplicate_full_paths"]).split("|") if item]
        if not paths and row["full_path"]:
            paths = [str(row["full_path"])]
        if not cfg.PROCESS_DUPLICATES:
            paths = paths[:1]
        for path in paths:
            documents.append({
                "customer_id": customer_id,
                "logical_document_type": str(row["expected_document_type"]),
                "pdf_path": Path(path),
                "is_duplicate": len(paths) > 1,
            })
    return documents


def process_customer(
    inventory: pd.DataFrame,
    customer_id: str,
    cfg: PipelineConfig = None,
    verbose: bool = True,
    call_budget: Optional[Dict[str, int]] = None,
) -> Dict[str, Any]:
    """RAW OCR of every target document of one customer. Never raises."""
    cfg = cfg or CFG
    started = time.time()

    documents = customer_documents(inventory, customer_id, cfg)
    missing = sorted(set(TARGET_DOCUMENTS) - {doc["logical_document_type"] for doc in documents})

    result: Dict[str, Any] = {
        "customer_id": customer_id,
        "documents_available": len(documents),
        "documents_missing": missing,
        "pdf_summaries": [],
        "records": [],
        "elapsed_s": 0.0,
    }

    if verbose:
        print("\n" + "#" * 78)
        print(f"# CUSTOMER {customer_id}")
        print("#" * 78)
        print(f"Target documents present : {len(documents)}")
        for _doc in documents:
            print(f"   - {_doc['logical_document_type']:28s} <- {_doc['pdf_path'].name}"
                  + ("   [duplicate set]" if _doc["is_duplicate"] else ""))
        print(f"Target documents missing : {len(missing)}")
        for _name in missing:
            print(f"   - {_name}")

    for document in documents:
        try:
            summary = process_pdf(
                document["pdf_path"],
                customer_id=customer_id,
                logical_document_type=document["logical_document_type"],
                cfg=cfg,
                verbose=verbose,
                call_budget=call_budget,
            )
        except Exception as exc:
            summary = {
                "customer_id": customer_id,
                "logical_document_type": document["logical_document_type"],
                "physical_filename": document["pdf_path"].name,
                "pdf_path": str(document["pdf_path"]),
                "status": "FAILED",
                "failure_type": "UNHANDLED_PDF_ERROR",
                "failure_message": repr(exc),
                "page_count": 0,
                "pages_processed": 0,
                "pages_reused": 0,
                "pages_skipped_blank": 0,
                "pages_skipped_budget": 0,
                "pages_failed": 0,
                "pages_degenerate": 0,
                "records": [],
                "elapsed_s": 0.0,
            }
            log(f"Unhandled PDF error {document['pdf_path'].name}: {exc!r}", logging.ERROR)

        result["pdf_summaries"].append(summary)
        result["records"].extend(summary.get("records", []))

    result["elapsed_s"] = round(time.time() - started, 3)

    if verbose:
        print(f"\n[CUSTOMER DONE] {customer_id} | PDFs={len(result['pdf_summaries'])} "
              f"| pages={len(result['records'])} | {result['elapsed_s']:.1f}s")

    return result


print("Single-customer processor ready: process_customer(), customer_documents()")

## 19 — Checkpointing and resumability

`dom.ipynb` checkpointed **one JSON per PDF**. RAW OCR needs finer granularity: a
30-page file that dies on page 28 must not lose 27 pages of GPU work. Checkpoints
are therefore **page-level**.

### Key

```text
sha256(
    customer_id | logical_document_type | physical_filename |
    pdf_sha256  | page_number           | pipeline_version
)
```

Including `pdf_sha256` means an edited or rescanned PDF invalidates its own
checkpoints. Including `physical_filename` means `FATCA.PDF` and `FATCA (1).PDF`
never collide. Including `customer_id` means two customers with identically named
documents never collide. Including `pipeline_version` means changing the prompt,
the rendering or the thresholds invalidates everything automatically — bump
`PIPELINE_VERSION` whenever a change should force a re-run.

Files live at `outputs/checkpoints/<customer_id>/<key>.json`, one directory per
customer, so a customer can be re-run by deleting one folder.

### Reuse policy

A page is skipped **only** when all four hold:

1. a checkpoint exists and parses;
2. its `pdf_sha256` matches the file on disk;
3. its `pipeline_version` matches;
4. its status is in `REUSABLE_STATUSES`.

`REUSABLE_STATUSES = {SUCCESS, SUCCESS_FALLBACK, BLANK_PAGE}`. **Failed and
degenerate pages are never reused** — re-running is precisely how you find out
whether a fix worked. `SKIPPED_BUDGET` is not reusable either: it is a
bookkeeping entry, not a result, and it is not even written to disk.

Writes are atomic (temp file + `os.replace`), so an interrupt during a write
cannot leave a half-written checkpoint behind.

In [ ]:
REUSABLE_STATUSES = {"SUCCESS", "SUCCESS_FALLBACK", "BLANK_PAGE"}
NON_PERSISTED_STATUSES = {"SKIPPED_BUDGET"}


def checkpoint_key(
    customer_id: str,
    logical_document_type: str,
    physical_filename: str,
    pdf_sha256: str,
    page_number: int,
    pipeline_version: str = PIPELINE_VERSION,
) -> str:
    """Deterministic, collision-free page identity."""
    raw = "|".join([
        str(customer_id),
        str(logical_document_type),
        str(physical_filename),
        str(pdf_sha256),
        str(int(page_number)),
        str(pipeline_version),
    ])
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()


def checkpoint_path(customer_id: str, key: str, cfg: PipelineConfig = None) -> Path:
    cfg = cfg or CFG
    safe_customer = re.sub(r"[^A-Za-z0-9._-]", "_", str(customer_id)) or "UNKNOWN"
    return cfg.CHECKPOINT_DIR / safe_customer / f"{key}.json"


def save_page_checkpoint(record: Dict[str, Any], cfg: PipelineConfig = None) -> Optional[Path]:
    """Persist one page result atomically. Bookkeeping statuses are not written."""
    cfg = cfg or CFG
    if record.get("status") in NON_PERSISTED_STATUSES:
        return None
    if not record.get("pdf_sha256"):
        return None  # benchmark records carry no hash and must not pollute the store

    key = checkpoint_key(
        record.get("customer_id"),
        record.get("logical_document_type"),
        record.get("physical_filename"),
        record.get("pdf_sha256"),
        record.get("page_number"),
        record.get("pipeline_version", PIPELINE_VERSION),
    )
    path = checkpoint_path(record.get("customer_id"), key, cfg)
    payload = dict(record)
    payload["checkpoint_key"] = key
    return write_json(path, payload)


def load_page_checkpoint(
    context: Dict[str, Any],
    page_number: int,
    cfg: PipelineConfig = None,
) -> Optional[Dict[str, Any]]:
    """
    Return a reusable checkpoint for this page, or None.

    All four reuse conditions must hold; anything else re-runs the page.
    """
    cfg = cfg or CFG
    pdf_sha256 = context.get("pdf_sha256")
    if not pdf_sha256:
        return None

    key = checkpoint_key(
        context.get("customer_id"),
        context.get("logical_document_type"),
        context.get("physical_filename"),
        pdf_sha256,
        page_number,
    )
    path = checkpoint_path(context.get("customer_id"), key, cfg)
    if not path.exists():
        return None

    try:
        record = json.loads(path.read_text(encoding="utf-8"))
    except Exception as exc:
        log(f"Corrupt checkpoint removed: {path.name} ({exc!r})", logging.WARNING)
        try:
            path.unlink()
        except Exception:
            pass
        return None

    if not isinstance(record, dict):
        return None
    if record.get("pipeline_version") != PIPELINE_VERSION:
        return None
    if record.get("pdf_sha256") != pdf_sha256:
        return None
    if record.get("status") not in REUSABLE_STATUSES:
        return None

    record["reused_from_checkpoint"] = True
    return record


def load_all_checkpoints(cfg: PipelineConfig = None) -> List[Dict[str, Any]]:
    """
    Every page record on disk for the current pipeline version.

    The checkpoint store is the source of truth: the JSONL / CSV / TXT outputs in
    section 24 are regenerated from it, which is what makes a resumed run produce
    exactly one line per page instead of duplicates.
    """
    cfg = cfg or CFG
    records: List[Dict[str, Any]] = []
    for path in sorted(cfg.CHECKPOINT_DIR.rglob("*.json")):
        try:
            record = json.loads(path.read_text(encoding="utf-8"))
        except Exception:
            continue
        if isinstance(record, dict) and record.get("pipeline_version") == PIPELINE_VERSION:
            records.append(record)

    records.sort(key=lambda item: (
        str(item.get("customer_id")),
        str(item.get("logical_document_type")),
        str(item.get("physical_filename")),
        int(item.get("page_number") or 0),
    ))
    return records


def checkpoint_store_stats(cfg: PipelineConfig = None) -> Dict[str, Any]:
    cfg = cfg or CFG
    records = load_all_checkpoints(cfg)
    by_status: Dict[str, int] = defaultdict(int)
    for record in records:
        by_status[str(record.get("status"))] += 1
    return {
        "records": len(records),
        "customers": len({str(record.get("customer_id")) for record in records}),
        "by_status": dict(by_status),
        "reusable": sum(1 for record in records if record.get("status") in REUSABLE_STATUSES),
    }


_store = checkpoint_store_stats(CFG)
print("Checkpoint utilities ready")
print(f"  directory        : {CFG.CHECKPOINT_DIR}")
print(f"  pipeline version : {PIPELINE_VERSION}")
print(f"  reusable statuses: {sorted(REUSABLE_STATUSES)}")
print(f"  records on disk  : {_store['records']} (from {_store['customers']} customer(s))")
if _store["by_status"]:
    for _status, _count in sorted(_store["by_status"].items()):
        print(f"      {_status:20s} {_count}")

## 20 — Diagnostic customer selection and call plan

Nothing is processed here. This cell:

1. picks `NUM_DIAGNOSTIC_CUSTOMERS` customer(s) **reproducibly** from
   `RANDOM_SEED` — the same seed always selects the same customers, so two runs
   are comparable;
2. lists their available target documents and counts their pages;
3. computes the **planned number of Qwen calls**;
4. compares that plan to `MAX_QWEN_CALLS` and to the remaining budget, and states
   plainly what will be skipped;
5. projects the runtime from the warm benchmark.

With `DIAGNOSTIC_PREFER_COMPLETE = True` the seeded draw is taken from the
customers with the **most** target documents present, which makes the first
diagnostic run cover as many of the five document types as possible.

In [ ]:
def select_diagnostic_customers(
    inventory: pd.DataFrame,
    cfg: PipelineConfig = None,
) -> List[str]:
    """Reproducible customer selection driven only by RANDOM_SEED."""
    cfg = cfg or CFG
    coverage = (
        inventory[inventory["exists"]]
        .groupby("customer_id")["expected_document_type"]
        .nunique()
        .sort_index()
    )
    if coverage.empty:
        return []

    eligible = sorted(coverage.index.tolist())

    if cfg.DIAGNOSTIC_PREFER_COMPLETE:
        best = int(coverage.max())
        preferred = sorted(coverage[coverage == best].index.tolist())
        pool = preferred if preferred else eligible
    else:
        pool = eligible

    generator = random.Random(cfg.RANDOM_SEED)
    count = min(int(cfg.NUM_DIAGNOSTIC_CUSTOMERS), len(pool))
    return sorted(generator.sample(pool, count))


def build_call_plan(
    inventory: pd.DataFrame,
    customer_ids: Sequence[str],
    cfg: PipelineConfig = None,
) -> Dict[str, Any]:
    """Count PDFs, pages and planned Qwen calls before spending a single second of GPU."""
    cfg = cfg or CFG
    plan: Dict[str, Any] = {
        "customers": list(customer_ids),
        "documents": [],
        "pdf_count": 0,
        "page_count": 0,
        "pages_already_checkpointed": 0,
        "pages_to_process": 0,
        "planned_qwen_calls": 0,
        "unreadable_pdfs": [],
    }

    for customer_id in customer_ids:
        for document in customer_documents(inventory, customer_id, cfg):
            pdf_path = document["pdf_path"]
            try:
                page_count = pdf_page_count(pdf_path)
                pdf_sha = sha256_file(pdf_path)
            except Exception as exc:
                plan["unreadable_pdfs"].append({"path": str(pdf_path), "error": repr(exc)})
                continue

            context = {
                "customer_id": customer_id,
                "logical_document_type": document["logical_document_type"],
                "physical_filename": pdf_path.name,
                "pdf_sha256": pdf_sha,
                "page_count": page_count,
            }
            already = sum(
                1 for page_number in range(1, page_count + 1)
                if load_page_checkpoint(context, page_number, cfg) is not None
            )

            plan["documents"].append({
                "customer_id": customer_id,
                "logical_document_type": document["logical_document_type"],
                "physical_filename": pdf_path.name,
                "pdf_path": str(pdf_path),
                "page_count": page_count,
                "already_checkpointed": already,
                "to_process": page_count - already,
                "is_duplicate": document["is_duplicate"],
            })
            plan["pdf_count"] += 1
            plan["page_count"] += page_count
            plan["pages_already_checkpointed"] += already
            plan["pages_to_process"] += page_count - already

    # One call per page is the design target; the HD fallback may add at most one.
    plan["planned_qwen_calls"] = plan["pages_to_process"]
    plan["worst_case_qwen_calls"] = (
        plan["pages_to_process"] * 2 if cfg.ENABLE_HD_FALLBACK else plan["pages_to_process"]
    )
    return plan


DIAGNOSTIC_CUSTOMERS = select_diagnostic_customers(INVENTORY_DF, CFG)
if not DIAGNOSTIC_CUSTOMERS:
    raise RuntimeError("No eligible customer found (no customer has any target document).")

CALL_PLAN = build_call_plan(INVENTORY_DF, DIAGNOSTIC_CUSTOMERS, CFG)

print("=" * 78)
print("DIAGNOSTIC CALL PLAN")
print("=" * 78)
print(f"Diagnostic mode          : {CFG.DIAGNOSTIC_MODE}")
print(f"Random seed              : {CFG.RANDOM_SEED}  (selection is reproducible)")
print(f"Prefer complete customers: {CFG.DIAGNOSTIC_PREFER_COMPLETE}")
print(f"Selected customer(s)     : {', '.join(DIAGNOSTIC_CUSTOMERS)}")
print("-" * 78)

for _customer in DIAGNOSTIC_CUSTOMERS:
    _rows = INVENTORY_DF[INVENTORY_DF["customer_id"] == _customer]
    _present = _rows[_rows["exists"]]
    _absent = _rows[~_rows["exists"]]
    print(f"Customer {_customer}:")
    print(f"  target documents present : {len(_present)}/{len(TARGET_DOCUMENTS)}")
    for _, _row in _present.iterrows():
        print(f"     + {_row['expected_document_type']:28s} {_row['matched_filename']}"
              + (f"   [{_row['duplicate_count']} copies]" if _row["duplicate_count"] > 1 else ""))
    for _, _row in _absent.iterrows():
        print(f"     - {_row['expected_document_type']:28s} MISSING")

print("-" * 78)
print("Per-PDF page counts:")
for _document in CALL_PLAN["documents"]:
    print(
        f"  {_document['physical_filename']:38s} pages={_document['page_count']:3d} "
        f"checkpointed={_document['already_checkpointed']:3d} "
        f"to_process={_document['to_process']:3d}"
    )
for _bad in CALL_PLAN["unreadable_pdfs"]:
    print(f"  UNREADABLE {_bad['path']} -> {_bad['error']}")

print("-" * 78)
print(f"PDFs in plan             : {CALL_PLAN['pdf_count']}")
print(f"Pages discovered         : {CALL_PLAN['page_count']}")
print(f"Pages already done       : {CALL_PLAN['pages_already_checkpointed']}")
print(f"Pages to process         : {CALL_PLAN['pages_to_process']}")
print(f"Planned Qwen calls       : {CALL_PLAN['planned_qwen_calls']}  (1 per page)")
print(f"Worst case with fallback : {CALL_PLAN['worst_case_qwen_calls']}")
print(f"MAX_QWEN_CALLS           : {CFG.MAX_QWEN_CALLS}")
print(f"Calls already used       : {QWEN_CALL_COUNTER} (benchmark)")

CALL_BUDGET = {"remaining": max(0, int(CFG.MAX_QWEN_CALLS))}
print(f"Budget for this run      : {CALL_BUDGET['remaining']}")

if CALL_PLAN["planned_qwen_calls"] > CALL_BUDGET["remaining"]:
    print()
    print("!" * 78)
    print(f"!!  The plan needs {CALL_PLAN['planned_qwen_calls']} calls but the budget is "
          f"{CALL_BUDGET['remaining']}.")
    print("!!  Pages beyond the budget will be recorded as SKIPPED_BUDGET, NOT processed.")
    print("!!  Raise CFG.MAX_QWEN_CALLS deliberately if you want the whole customer.")
    print("!" * 78)

_warm_estimate = float(BENCH_WARM["total_time_s"])
_pages_expected = min(CALL_PLAN["pages_to_process"], CALL_BUDGET["remaining"])
print("-" * 78)
print(f"Projected runtime        : {format_duration(_warm_estimate * _pages_expected)} "
      f"({_pages_expected} pages x {_warm_estimate:.1f}s warm)")
print("=" * 78)

log(
    f"Call plan: customers={DIAGNOSTIC_CUSTOMERS} pdfs={CALL_PLAN['pdf_count']} "
    f"pages={CALL_PLAN['page_count']} to_process={CALL_PLAN['pages_to_process']} "
    f"budget={CALL_BUDGET['remaining']}"
)

## 21 — Diagnostic run

The first cell that performs a real multi-page run. It processes only the
customer(s) selected in §20, under the call budget computed there.

Re-running this cell after a kernel restart is safe and cheap: completed pages
are served from checkpoints and no GPU work is repeated.

If `CFG.DIAGNOSTIC_MODE` has been switched off (because you moved on to the
full-dataset run in §25), this cell prints a notice and does nothing — it does
not fail — so the notebook still runs top to bottom in either mode.

In [ ]:
DIAGNOSTIC_RUN_SKIPPED = not CFG.DIAGNOSTIC_MODE

RUN_STARTED_AT = datetime.now()
_run_t0 = time.time()

CUSTOMER_RESULTS = []
RUN_RECORDS = []
RUN_ERRORS = []
PDF_FAILURES = []

if DIAGNOSTIC_RUN_SKIPPED:
    # Not an error: the notebook was switched to full-dataset mode, so this
    # guarded single-customer run is deliberately bypassed. Sections 22-27 stay
    # runnable and will report on whatever the checkpoint store contains.
    print("=" * 78)
    print("DIAGNOSTIC RUN SKIPPED")
    print("=" * 78)
    print("CFG.DIAGNOSTIC_MODE is False, so this cell does nothing.")
    print("Use section 25 for the full-dataset run, then continue with sections 22-27.")
    print("=" * 78)
    _customers_to_run = []
else:
    print("=" * 78)
    print(f"DIAGNOSTIC RUN | {PIPELINE_VERSION}")
    print(f"Started {RUN_STARTED_AT.isoformat(timespec='seconds')}")
    print(f"Customers: {', '.join(DIAGNOSTIC_CUSTOMERS)} | budget: {CALL_BUDGET['remaining']} Qwen calls")
    print("=" * 78)
    _customers_to_run = list(DIAGNOSTIC_CUSTOMERS)

for _position, _customer_id in enumerate(_customers_to_run, 1):
    print(f"\n>>> [{_position}/{len(DIAGNOSTIC_CUSTOMERS)}] customer {_customer_id}")
    try:
        _result = process_customer(
            INVENTORY_DF,
            _customer_id,
            cfg=CFG,
            verbose=True,
            call_budget=CALL_BUDGET,
        )
        CUSTOMER_RESULTS.append(_result)
        RUN_RECORDS.extend(_result["records"])
        for _pdf_summary in _result.get("pdf_summaries", []):
            if _pdf_summary.get("status") == "FAILED":
                PDF_FAILURES.append({
                    "customer_id": _pdf_summary.get("customer_id"),
                    "logical_document_type": _pdf_summary.get("logical_document_type"),
                    "physical_filename": _pdf_summary.get("physical_filename"),
                    "failure_type": _pdf_summary.get("failure_type"),
                    "failure_message": _pdf_summary.get("failure_message"),
                })
    except Exception as exc:
        RUN_ERRORS.append({
            "customer_id": _customer_id,
            "stage": "PROCESS_CUSTOMER",
            "error": repr(exc),
            "at": datetime.now().isoformat(timespec="seconds"),
        })
        print(f"[ERROR] customer {_customer_id}: {exc!r}")
        log(f"Customer failed: {_customer_id} -> {exc!r}", logging.ERROR)
    finally:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

RUN_ELAPSED_S = round(time.time() - _run_t0, 2)

print("\n" + "=" * 78)
print("DIAGNOSTIC RUN FINISHED" if not DIAGNOSTIC_RUN_SKIPPED else "DIAGNOSTIC RUN NOT EXECUTED")
print("=" * 78)
print(f"Customers processed : {len(CUSTOMER_RESULTS)}")
print(f"Page records        : {len(RUN_RECORDS)}")
print(f"Customer errors     : {len(RUN_ERRORS)}")
print(f"PDFs that failed to open : {len(PDF_FAILURES)}")
print(f"Qwen calls total    : {QWEN_CALL_COUNTER} (includes the 2 benchmark calls)")
print(f"Budget remaining    : {CALL_BUDGET['remaining']}")
print(f"Wall clock          : {format_duration(RUN_ELAPSED_S)} ({RUN_ELAPSED_S}s)")

_status_counter: Dict[str, int] = defaultdict(int)
for _record in RUN_RECORDS:
    _status_counter[str(_record.get("status"))] += 1
print("-" * 78)
print("Status breakdown:")
for _status, _count in sorted(_status_counter.items()):
    print(f"  {_status:20s} {_count}")
print("=" * 78)

if PDF_FAILURES:
    print("-" * 78)
    print("PDFs that could not be opened (no page was lost silently):")
    for _failure in PDF_FAILURES:
        print(f"  {_failure['customer_id']} | {_failure['logical_document_type']} | "
              f"{_failure['physical_filename']} | {_failure['failure_type']}")
        print(f"      {_failure['failure_message']}")

for _error in RUN_ERRORS:
    print(f"ERROR {_error['customer_id']} | {_error['stage']} | {_error['error']}")

log(f"Diagnostic run finished: {len(RUN_RECORDS)} page records in {RUN_ELAPSED_S}s")

## 22 — RAW OCR inspection

Page by page: the transcription as the model produced it, with its degeneration
diagnostics next to it. This is the cell you actually read to answer the
question the notebook exists for.

Nothing here parses, normalises or interprets the text. It is printed raw.

In [ ]:
def inspect_records(
    records: Sequence[Dict[str, Any]],
    max_chars: int = None,
    show_blank: bool = False,
) -> None:
    """Print each page's RAW transcription with its diagnostics. No interpretation."""
    max_chars = CFG.PRINT_TRANSCRIPTION_MAX_CHARS if max_chars is None else max_chars

    if not records:
        print("No page record to inspect.")
        return

    for record in records:
        status = record.get("status")
        if status == "BLANK_PAGE" and not show_blank:
            print(
                f"\n[{record.get('customer_id')}] {record.get('logical_document_type')} "
                f"p{record.get('page_number')}/{record.get('page_count')} -> BLANK_PAGE (no Qwen call)"
            )
            continue

        print("\n" + "=" * 78)
        print(f"Customer      : {record.get('customer_id')}")
        print(f"Document      : {record.get('logical_document_type')}")
        print(f"Physical file : {record.get('physical_filename')}")
        print(f"Page          : {record.get('page_number')}/{record.get('page_count')}")
        print(f"Status        : {status}"
              + (f"   ({record.get('failure_type')})" if record.get("failure_type") else ""))
        print(f"Fallback used : {record.get('fallback_used')}   Qwen calls: {record.get('qwen_calls')}")
        print(f"Stop reason   : {record.get('stop_reason')}")
        print(f"Tokens in/out : {record.get('tokens_in')} / {record.get('tokens_out')}"
              f"   ({record.get('tokens_per_second')} tok/s)")
        print(f"Times (s)     : render={record.get('render_time_s')} "
              f"prep={record.get('preprocess_time_s')} "
              f"proc={record.get('processor_time_s')} "
              f"gen={record.get('generation_time_s')} "
              f"dec={record.get('decode_time_s')} "
              f"total={record.get('total_time_s')}")
        print(f"Image         : PIL {record.get('processed_width')}x{record.get('processed_height')} "
              f"| model {record.get('model_image_width')}x{record.get('model_image_height')} "
              f"| vision tokens {record.get('vision_tokens')}")

        diagnostics = record.get("diagnostics") or {}
        print(f"Degenerate    : {record.get('degenerate')} ({record.get('degenerate_reason')})")
        if diagnostics:
            print(
                f"  length={diagnostics.get('text_length')} "
                f"unique={diagnostics.get('unique_character_count')} "
                f"unique_ratio={diagnostics.get('unique_character_ratio')} "
                f"entropy={diagnostics.get('shannon_entropy_bits_per_char')} "
                f"punct={diagnostics.get('punctuation_ratio')} "
                f"max_run={diagnostics.get('longest_repeated_character_run')} "
                f"mrz_lines={diagnostics.get('mrz_line_count')} "
                f"line_repeat={diagnostics.get('max_consecutive_line_repeat')}"
            )

        preprocessing = record.get("preprocessing") or {}
        if isinstance(preprocessing, dict) and preprocessing.get("reverted"):
            print(f"Preprocessing : REVERTED ({preprocessing.get('revert_reason')})")

        if record.get("failure_message"):
            print(f"Failure       : {record.get('failure_message')}")

        text = record.get("raw_ocr") or ""
        print("-" * 78)
        if not text.strip():
            print("(no transcription)")
        else:
            print(text[:max_chars])
            if len(text) > max_chars:
                print(f"\n... [truncated for display: {len(text) - max_chars} more characters, "
                      f"full text is in the JSONL / TXT outputs]")
        print("=" * 78)


# Prefer this session's records; fall back to the checkpoint store when the
# diagnostic run was skipped (full-dataset mode) or resumed entirely from disk.
INSPECT_RECORDS = list(RUN_RECORDS)
INSPECT_SOURCE = "this run"
if not INSPECT_RECORDS:
    INSPECT_RECORDS = load_all_checkpoints(CFG)[:25]
    INSPECT_SOURCE = "checkpoint store (first 25)"

print(f"Inspecting {len(INSPECT_RECORDS)} page record(s) from {INSPECT_SOURCE}.")

if CFG.PRINT_TRANSCRIPTION:
    inspect_records(INSPECT_RECORDS)
else:
    print("CFG.PRINT_TRANSCRIPTION is False - transcriptions are only written to disk.")
    print("Call inspect_records(INSPECT_RECORDS) manually to read them here.")

## 23 — Performance analysis and root-cause attribution

A single total runtime is useless for debugging. This cell reports every stage
independently and names the dominant cost, so "OCR is slow" becomes a specific,
actionable finding:

| Observation | Root cause | Action |
|---|---|---|
| `generation_time_s` dominates **and** tok/s ≈ 25–40 | normal: decode is bandwidth-bound | reduce `max_new_tokens`, or accept it |
| `generation_time_s` dominates **and** tok/s < 8 | CPU offload, or a thermally/power-capped GPU | §11 device map |
| many pages stop on `MAX_TOKENS` | `max_new_tokens` too low, or degenerate output | inspect the text before raising the cap |
| `processor_time_s` is large | images too big — the processor is resizing and patching on CPU | lower `IMAGE_MAX_SIZE_STANDARD` |
| `render_time_s` is large | `RENDER_ZOOM` too high for the source DPI | lower `RENDER_ZOOM_STANDARD` |
| `preprocess_time_s` is large | deskew search is the usual suspect | widen `DESKEW_STEP` or disable |
| `device_transfer_time_s` is large | oversized pixel tensors crossing PCIe | lower the image size |
| high `fallback_used` rate | first-pass quality problem, and it **doubles** cost | fix quality, do not tune the fallback |

In [ ]:
PERFORMANCE_COLUMNS = [
    "customer_id", "logical_document_type", "physical_filename", "pdf_sha256",
    "page_number", "page_count",
    "render_time_s", "preprocess_time_s", "template_time_s", "processor_time_s",
    "device_transfer_time_s", "generation_time_s", "decode_time_s",
    "validation_time_s", "total_time_s",
    "render_width", "render_height", "processed_width", "processed_height",
    "model_image_width", "model_image_height", "vision_tokens",
    "white_ratio", "ink_ratio",
    "tokens_in", "tokens_out", "tokens_per_second", "stop_reason", "max_new_tokens",
    "gpu_memory_allocated_before_mb", "gpu_memory_allocated_after_mb", "gpu_memory_reserved_mb",
    "status", "failure_type", "failure_message", "fallback_used", "qwen_calls",
    "degenerate", "degenerate_reason", "processed_at",
]

STAGE_COLUMNS = [
    "render_time_s", "preprocess_time_s", "template_time_s", "processor_time_s",
    "device_transfer_time_s", "generation_time_s", "decode_time_s", "validation_time_s",
]


def records_to_dataframe(records: Sequence[Dict[str, Any]]) -> pd.DataFrame:
    """Flat performance table; missing keys become NaN rather than exploding."""
    rows = []
    for record in records:
        rows.append({column: record.get(column) for column in PERFORMANCE_COLUMNS})
    frame = pd.DataFrame(rows, columns=PERFORMANCE_COLUMNS)
    for column in STAGE_COLUMNS + ["tokens_in", "tokens_out", "tokens_per_second", "vision_tokens"]:
        frame[column] = pd.to_numeric(frame[column], errors="coerce")
    return frame


def percentile(values: Sequence[float], fraction: float) -> float:
    cleaned = sorted(float(value) for value in values if value is not None and not pd.isna(value))
    if not cleaned:
        return 0.0
    if len(cleaned) == 1:
        return cleaned[0]
    position = fraction * (len(cleaned) - 1)
    low = int(math.floor(position))
    high = int(math.ceil(position))
    if low == high:
        return cleaned[low]
    return cleaned[low] + (cleaned[high] - cleaned[low]) * (position - low)


def analyse_performance(records: Sequence[Dict[str, Any]]) -> Dict[str, Any]:
    """Stage breakdown, percentiles and an explicit bottleneck verdict."""
    frame = records_to_dataframe(records)
    inferred = frame[frame["status"].isin(["SUCCESS", "SUCCESS_FALLBACK", "DEGENERATE"])]

    analysis: Dict[str, Any] = {
        "pages_total": int(len(frame)),
        "pages_inferred": int(len(inferred)),
        "stage_totals": {},
        "stage_means": {},
        "bottleneck": None,
        "bottleneck_share": 0.0,
    }

    if inferred.empty:
        print("No inferred page to analyse (all pages were blank, reused, skipped or failed).")
        return analysis

    total_pipeline_time = float(inferred[STAGE_COLUMNS].sum().sum())
    for column in STAGE_COLUMNS:
        total = float(inferred[column].sum())
        analysis["stage_totals"][column] = round(total, 3)
        analysis["stage_means"][column] = round(float(inferred[column].mean()), 4)

    if total_pipeline_time > 0:
        bottleneck = max(analysis["stage_totals"].items(), key=lambda item: item[1])
        analysis["bottleneck"] = bottleneck[0]
        analysis["bottleneck_share"] = round(100.0 * bottleneck[1] / total_pipeline_time, 1)

    generation = inferred["generation_time_s"].dropna().tolist()
    totals = inferred["total_time_s"].dropna().tolist()
    throughput = inferred["tokens_per_second"].dropna().tolist()

    analysis.update({
        "generation_mean_s": round(float(np.mean(generation)), 3) if generation else 0.0,
        "generation_median_s": round(float(statistics.median(generation)), 3) if generation else 0.0,
        "generation_p95_s": round(percentile(generation, 0.95), 3),
        "total_mean_s": round(float(np.mean(totals)), 3) if totals else 0.0,
        "total_median_s": round(float(statistics.median(totals)), 3) if totals else 0.0,
        "total_p95_s": round(percentile(totals, 0.95), 3),
        "tokens_per_second_mean": round(float(np.mean(throughput)), 2) if throughput else 0.0,
        "tokens_in_total": int(inferred["tokens_in"].fillna(0).sum()),
        "tokens_out_total": int(inferred["tokens_out"].fillna(0).sum()),
        "vision_tokens_mean": round(float(inferred["vision_tokens"].dropna().mean()), 1)
        if inferred["vision_tokens"].notna().any() else None,
        "max_tokens_hits": int((inferred["stop_reason"] == "MAX_TOKENS").sum()),
        "repetition_guard_hits": int((inferred["stop_reason"] == "REPETITION_GUARD").sum()),
        "fallback_pages": int(inferred["fallback_used"].fillna(False).astype(bool).sum()),
        "degenerate_pages": int(inferred["degenerate"].fillna(False).astype(bool).sum()),
    })

    # ------------------------------------------------------------- printing --
    print("=" * 78)
    print("PERFORMANCE ANALYSIS")
    print("=" * 78)
    print(f"Pages in table          : {analysis['pages_total']}")
    print(f"Pages actually inferred : {analysis['pages_inferred']}")
    print("-" * 78)
    print(f"{'stage':26s} {'total (s)':>11s} {'mean (s)':>10s} {'share':>8s}")
    print("-" * 78)
    for column in STAGE_COLUMNS:
        total = analysis["stage_totals"][column]
        share = (100.0 * total / total_pipeline_time) if total_pipeline_time > 0 else 0.0
        print(f"{column:26s} {total:11.3f} {analysis['stage_means'][column]:10.4f} {share:7.1f}%")
    print("-" * 78)
    print(f"{'measured pipeline time':26s} {total_pipeline_time:11.3f}")
    print(f"Dominant stage          : {analysis['bottleneck']} ({analysis['bottleneck_share']}% of measured time)")
    print("-" * 78)
    print(f"Generation  mean/median/p95 : {analysis['generation_mean_s']:.2f} / "
          f"{analysis['generation_median_s']:.2f} / {analysis['generation_p95_s']:.2f} s")
    print(f"Total page  mean/median/p95 : {analysis['total_mean_s']:.2f} / "
          f"{analysis['total_median_s']:.2f} / {analysis['total_p95_s']:.2f} s")
    print(f"Output throughput mean      : {analysis['tokens_per_second_mean']:.2f} tok/s")
    print(f"Tokens in / out             : {analysis['tokens_in_total']:,} / {analysis['tokens_out_total']:,}")
    print(f"Vision tokens mean          : {analysis['vision_tokens_mean']}")
    print(f"Stopped on MAX_TOKENS       : {analysis['max_tokens_hits']}")
    print(f"Stopped by repetition guard : {analysis['repetition_guard_hits']}")
    print(f"Pages using HD fallback     : {analysis['fallback_pages']}")
    print(f"Degenerate pages            : {analysis['degenerate_pages']}")
    print("=" * 78)

    # -------------------------------------------------------- verdict --------
    print("ROOT-CAUSE READING")
    print("-" * 78)
    verdicts: List[str] = []

    if analysis["bottleneck"] == "generation_time_s":
        if analysis["tokens_per_second_mean"] < CFG.LOW_THROUGHPUT_WARNING_TPS:
            verdicts.append(
                f"Generation dominates AND throughput is only {analysis['tokens_per_second_mean']:.1f} tok/s. "
                "That is abnormal for a 27B bf16 model on an H100 (expect 25-40). "
                "Check section 11 for CPU/disk offload before tuning anything else."
            )
        else:
            verdicts.append(
                f"Generation dominates at a normal {analysis['tokens_per_second_mean']:.1f} tok/s: "
                "decode is memory-bandwidth bound and this is expected. "
                "The only real lever is fewer output tokens (max_new_tokens, or shorter pages)."
            )
    elif analysis["bottleneck"] == "processor_time_s":
        verdicts.append(
            "The processor dominates: images are large and patching/resizing happens on CPU. "
            "Lower IMAGE_MAX_SIZE_STANDARD."
        )
    elif analysis["bottleneck"] == "render_time_s":
        verdicts.append(
            "PDF rendering dominates: RENDER_ZOOM_STANDARD is high relative to the source DPI. "
            "Lower it; the image is downscaled to IMAGE_MAX_SIZE_STANDARD afterwards anyway."
        )
    elif analysis["bottleneck"] == "preprocess_time_s":
        verdicts.append(
            "Preprocessing dominates: the deskew angle search is the usual cause. "
            "Raise DESKEW_STEP or set ENABLE_DESKEW=False."
        )
    elif analysis["bottleneck"] == "device_transfer_time_s":
        verdicts.append("Host-to-device transfer dominates: the pixel tensors are oversized.")

    if analysis["max_tokens_hits"]:
        verdicts.append(
            f"{analysis['max_tokens_hits']} page(s) hit max_new_tokens. Read those transcriptions: "
            "genuinely dense pages justify a higher cap, degenerate ones do not."
        )
    if analysis["repetition_guard_hits"]:
        verdicts.append(
            f"The repetition guard fired on {analysis['repetition_guard_hits']} page(s). "
            "Without it each would have run to max_new_tokens - that is the '!!!!!!!!' cost."
        )
    if analysis["fallback_pages"]:
        share = 100.0 * analysis["fallback_pages"] / max(analysis["pages_inferred"], 1)
        verdicts.append(
            f"{analysis['fallback_pages']} page(s) ({share:.0f}%) needed the HD fallback, "
            "which doubles their cost. Fix first-pass quality rather than the fallback."
        )
    if CPU_OFFLOAD_DETECTED:
        verdicts.append("CPU/disk offload is active: every figure above is contaminated.")

    if not verdicts:
        verdicts.append("Nothing anomalous: timings are dominated by normal decode cost.")

    for _index, _verdict in enumerate(verdicts, 1):
        print(f"  {_index}. {_verdict}")
    print("=" * 78)

    analysis["verdicts"] = verdicts
    analysis["frame"] = frame
    return analysis


ALL_RECORDS = load_all_checkpoints(CFG)
print(f"Records loaded from the checkpoint store: {len(ALL_RECORDS)}")
print(f"(RUN_RECORDS from this session: {len(RUN_RECORDS)})\n")

PERFORMANCE = analyse_performance(ALL_RECORDS if ALL_RECORDS else RUN_RECORDS)
PERFORMANCE_DF = PERFORMANCE.get("frame", records_to_dataframe(RUN_RECORDS))

## 24 — Output files

```text
outputs/
├── inventory/     kyc_document_inventory.csv | .json
├── raw_ocr/       raw_ocr_results.jsonl | .csv | .txt
├── performance/   raw_ocr_performance.csv
├── checkpoints/   <customer_id>/<page_key>.json
└── logs/          kyc_raw_ocr.log
```

The JSONL has **one record per page**. It is regenerated from the checkpoint
store rather than appended to during the run — appending would produce duplicate
lines on every resume, whereas regenerating from the store always yields exactly
one line per page.

All serialisation goes through `to_jsonable()` / `json_default()`, so NumPy
scalars, NumPy arrays and torch tensors cannot break a write. The CSV keeps the
full transcription (quoted, newlines preserved) plus a single-line preview column
for spreadsheet readability.

In [ ]:
JSONL_FIELDS = [
    "customer_id", "logical_document_type", "physical_filename", "pdf_sha256",
    "page_number", "page_count", "raw_ocr", "status", "degenerate", "fallback_used",
    "tokens_in", "tokens_out", "generation_time_s", "total_time_s",
]


def write_raw_ocr_outputs(
    records: Sequence[Dict[str, Any]],
    cfg: PipelineConfig = None,
) -> Dict[str, Path]:
    """
    Regenerate every RAW OCR output from the given records.

    Regeneration (rather than appending) is what guarantees one line per page
    after a resumed run.
    """
    cfg = cfg or CFG
    jsonl_path = cfg.RAW_OCR_DIR / "raw_ocr_results.jsonl"
    csv_path = cfg.RAW_OCR_DIR / "raw_ocr_results.csv"
    text_path = cfg.RAW_OCR_DIR / "raw_ocr_results.txt"
    performance_path = cfg.PERFORMANCE_DIR / "raw_ocr_performance.csv"

    # ------------------------------------------------------------- JSONL ----
    temporary = jsonl_path.with_suffix(".jsonl.tmp")
    with open(temporary, "w", encoding="utf-8") as handle:
        for record in records:
            payload = {field: record.get(field) for field in JSONL_FIELDS}
            payload["degenerate_reason"] = record.get("degenerate_reason")
            payload["stop_reason"] = record.get("stop_reason")
            payload["failure_type"] = record.get("failure_type")
            payload["pipeline_version"] = record.get("pipeline_version", PIPELINE_VERSION)
            handle.write(
                json.dumps(to_jsonable(payload), ensure_ascii=False, default=json_default) + "\n"
            )
    os.replace(temporary, jsonl_path)

    # --------------------------------------------------------------- CSV ----
    csv_rows = []
    for record in records:
        text = record.get("raw_ocr") or ""
        csv_rows.append({
            "customer_id": record.get("customer_id"),
            "logical_document_type": record.get("logical_document_type"),
            "physical_filename": record.get("physical_filename"),
            "pdf_sha256": record.get("pdf_sha256"),
            "page_number": record.get("page_number"),
            "page_count": record.get("page_count"),
            "status": record.get("status"),
            "degenerate": record.get("degenerate"),
            "degenerate_reason": record.get("degenerate_reason"),
            "fallback_used": record.get("fallback_used"),
            "stop_reason": record.get("stop_reason"),
            "tokens_in": record.get("tokens_in"),
            "tokens_out": record.get("tokens_out"),
            "generation_time_s": record.get("generation_time_s"),
            "total_time_s": record.get("total_time_s"),
            "character_count": len(text),
            "raw_ocr_preview": re.sub(r"\s+", " ", text)[:300],
            "raw_ocr": text,
        })
    pd.DataFrame(csv_rows).to_csv(csv_path, index=False, encoding="utf-8-sig")

    # --------------------------------------------------------------- TXT ----
    with open(text_path, "w", encoding="utf-8") as handle:
        handle.write(f"{PIPELINE_VERSION}\n")
        handle.write(f"generated_at: {datetime.now().isoformat(timespec='seconds')}\n")
        handle.write(f"pages: {len(records)}\n\n")
        for record in records:
            handle.write("=" * 78 + "\n")
            handle.write(f"customer_id   : {record.get('customer_id')}\n")
            handle.write(f"document type : {record.get('logical_document_type')}\n")
            handle.write(f"physical file : {record.get('physical_filename')}\n")
            handle.write(f"page          : {record.get('page_number')}/{record.get('page_count')}\n")
            handle.write(f"status        : {record.get('status')}\n")
            handle.write(f"degenerate    : {record.get('degenerate')} ({record.get('degenerate_reason')})\n")
            handle.write("=" * 78 + "\n")
            handle.write((record.get("raw_ocr") or "(no transcription)") + "\n\n")

    # ------------------------------------------------------- performance ----
    records_to_dataframe(records).to_csv(performance_path, index=False, encoding="utf-8-sig")

    return {
        "jsonl": jsonl_path,
        "csv": csv_path,
        "txt": text_path,
        "performance": performance_path,
    }


OUTPUT_PATHS = write_raw_ocr_outputs(ALL_RECORDS if ALL_RECORDS else RUN_RECORDS, CFG)

print("Outputs written")
print("-" * 78)
for _label, _path in OUTPUT_PATHS.items():
    _size = _path.stat().st_size if _path.exists() else 0
    print(f"  {_label:12s} {str(_path):64s} {_size:>10,} bytes")
print(f"  {'inventory':12s} {str(INVENTORY_CSV_PATH):64s} "
      f"{INVENTORY_CSV_PATH.stat().st_size:>10,} bytes")
print(f"  {'log':12s} {str(CFG.LOG_PATH):64s} "
      f"{CFG.LOG_PATH.stat().st_size:>10,} bytes")
print("-" * 78)
print(f"Records written: {len(ALL_RECORDS) if ALL_RECORDS else len(RUN_RECORDS)}")

log(f"Outputs written to {CFG.OUTPUT_DIR}")

## 25 — Full-dataset runner (disabled by default)

**This cell does nothing until you deliberately enable it.** Two gates must both
be opened:

```python
CFG.RUN_FULL_DATASET = True
CFG.DIAGNOSTIC_MODE  = False
```

Before flipping them, you should have:

1. read the transcriptions in §22 and judged the OCR usable;
2. read the stage breakdown in §23 and found no anomaly;
3. seen a warm page time you are willing to multiply by the whole archive.

The cell prints the projected total and the full call count **before** starting,
and raises `FULL_RUN_MAX_QWEN_CALLS` as an explicit, separate budget — the
diagnostic `MAX_QWEN_CALLS = 20` is not silently reused.

Resumability makes a large run interruptible: stop the kernel whenever you like
and re-run: completed pages come back from checkpoints.

In [ ]:
# -----------------------------------------------------------------------------
# FULL DATASET RUNNER - DISABLED BY DEFAULT
#
# To enable:
#     CFG.RUN_FULL_DATASET = True
#     CFG.DIAGNOSTIC_MODE  = False
#     FULL_RUN_MAX_QWEN_CALLS = <an explicit number you have decided on>
# -----------------------------------------------------------------------------

FULL_RUN_MAX_QWEN_CALLS = 100_000       # explicit, separate from the diagnostic budget
FULL_RUN_VERBOSE = False                # per-page printing is noise at scale
FULL_RUN_PROGRESS_EVERY = 1             # print one line per customer


def run_full_dataset(
    inventory: pd.DataFrame,
    cfg: PipelineConfig = None,
    max_calls: int = FULL_RUN_MAX_QWEN_CALLS,
    verbose: bool = FULL_RUN_VERBOSE,
) -> Dict[str, Any]:
    """Process every customer in the inventory, resumable and error-isolated."""
    cfg = cfg or CFG
    started = time.time()
    customer_ids = sorted(inventory["customer_id"].unique().tolist())
    budget = {"remaining": int(max_calls)}

    results: List[Dict[str, Any]] = []
    errors: List[Dict[str, Any]] = []

    print("=" * 78)
    print(f"FULL DATASET RUN | {PIPELINE_VERSION}")
    print(f"Customers: {len(customer_ids)} | call budget: {budget['remaining']}")
    print("=" * 78)

    for position, customer_id in enumerate(customer_ids, 1):
        try:
            result = process_customer(
                inventory, customer_id, cfg=cfg, verbose=verbose, call_budget=budget
            )
            results.append(result)
        except Exception as exc:
            errors.append({
                "customer_id": customer_id,
                "error": repr(exc),
                "at": datetime.now().isoformat(timespec="seconds"),
            })
            log(f"Customer failed: {customer_id} -> {exc!r}", logging.ERROR)
        finally:
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        if position % max(1, FULL_RUN_PROGRESS_EVERY) == 0:
            elapsed = time.time() - started
            average = elapsed / position
            eta = average * (len(customer_ids) - position)
            pages_done = sum(len(item["records"]) for item in results)
            print(
                f"[{position}/{len(customer_ids)}] {customer_id} | "
                f"pages={pages_done} | calls_left={budget['remaining']} | "
                f"elapsed={format_duration(elapsed)} | ETA={format_duration(eta)}",
                flush=True,
            )

        if budget["remaining"] <= 0:
            print(f"Call budget exhausted after {position} customer(s). Stopping.")
            break

    elapsed = round(time.time() - started, 2)
    records = load_all_checkpoints(cfg)
    paths = write_raw_ocr_outputs(records, cfg)

    print("=" * 78)
    print(f"FULL RUN FINISHED in {format_duration(elapsed)}")
    print(f"Customers processed : {len(results)} / {len(customer_ids)}")
    print(f"Errors              : {len(errors)}")
    print(f"Page records on disk: {len(records)}")
    print(f"Outputs             : {paths['jsonl']}")
    print("=" * 78)

    return {
        "customers": results,
        "errors": errors,
        "records": records,
        "paths": paths,
        "elapsed_s": elapsed,
    }


if CFG.RUN_FULL_DATASET and not CFG.DIAGNOSTIC_MODE:
    _total_pages_estimate = 0
    for _, _row in INVENTORY_DF[INVENTORY_DF["exists"]].iterrows():
        try:
            _total_pages_estimate += pdf_page_count(Path(_row["full_path"]))
        except Exception:
            pass

    _warm = float(BENCH_WARM["total_time_s"])
    print("PRE-FLIGHT")
    print(f"  PDFs            : {int(INVENTORY_DF['exists'].sum())}")
    print(f"  Pages estimated : {_total_pages_estimate}")
    print(f"  Warm page time  : {_warm:.1f} s")
    print(f"  Projected total : {format_duration(_warm * _total_pages_estimate)}")
    print()
    FULL_RUN_RESULT = run_full_dataset(INVENTORY_DF, CFG)
else:
    FULL_RUN_RESULT = None
    print("Full dataset run is DISABLED.")
    print(f"  CFG.RUN_FULL_DATASET = {CFG.RUN_FULL_DATASET}")
    print(f"  CFG.DIAGNOSTIC_MODE  = {CFG.DIAGNOSTIC_MODE}")
    print("  Set RUN_FULL_DATASET=True and DIAGNOSTIC_MODE=False, then re-run this cell.")

## 26 — Optional experimental batching (disabled)

Batch size 1 is the baseline, and it stays the baseline until single-page
correctness is proven. Batching is an optimisation for a pipeline that already
produces correct output — turning it on first only means being wrong faster.

There are also two concrete risks specific to this workload:

* **Padding waste.** KYC pages vary in size, so their vision-token counts vary.
  In a batch every sequence is padded to the longest, and decoding continues
  until the *last* sequence finishes — one dense page makes the whole batch pay
  its length.
* **VRAM.** A 27 B bf16 model already occupies ≈ 54 GB of an 80 GB H100. Two
  concurrent 2 000-vision-token sequences plus their KV caches is where OOM
  starts, and an OOM mid-batch loses every page in that batch, not one.

The function below follows the proven `ask_batch` shape from `dom.ipynb`
(left padding, per-sequence attention-mask token counts, per-sequence input
trimming before decode). It is **not called anywhere**. Enable it only after the
baseline is validated, and measure against batch size 1 rather than assuming.

In [ ]:
ENABLE_EXPERIMENTAL_BATCHING = False   # keep False until the baseline is validated


def qwen_batch_call(
    prompt: str,
    images: Sequence[Image.Image],
    max_new_tokens: int,
) -> List[Dict[str, Any]]:
    """
    EXPERIMENTAL, NOT USED BY THE PIPELINE.

    Batched multimodal generation following the dom.ipynb ask_batch() contract:
    left padding, per-sequence token counts from the attention mask, and
    per-sequence input trimming before decode.
    """
    if not ENABLE_EXPERIMENTAL_BATCHING:
        raise RuntimeError(
            "Batching is disabled. Validate single-page correctness first, then set "
            "ENABLE_EXPERIMENTAL_BATCHING = True deliberately."
        )
    if not images:
        return []
    if len(images) == 1:
        return [qwen_single_call(prompt, images[0], max_new_tokens)]

    texts_in = [apply_template(build_messages(prompt, image)) for image in images]

    inputs = processor(
        text=texts_in,
        images=list(images),
        return_tensors="pt",
        padding=True,
    ).to(INPUT_DEVICE)

    torch.cuda.synchronize()
    started = time.time()
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=int(max_new_tokens),
            do_sample=False,
            repetition_penalty=CFG.REPETITION_PENALTY,
            pad_token_id=processor.tokenizer.eos_token_id,
        )
    torch.cuda.synchronize()
    elapsed = time.time() - started

    if output_ids.shape[0] != len(images):
        raise RuntimeError(
            f"Inconsistent batch output: {output_ids.shape[0]} sequence(s) for {len(images)} image(s)"
        )

    input_width = int(inputs["input_ids"].shape[1])
    attention_mask = inputs.get("attention_mask")

    results: List[Dict[str, Any]] = []
    for index in range(len(images)):
        generated = output_ids[index][input_width:]
        text = processor.decode(
            generated, skip_special_tokens=True, clean_up_tokenization_spaces=True
        )
        tokens_in = (
            int(attention_mask[index].sum().item()) if attention_mask is not None else input_width
        )
        tokens_out = int(generated.shape[0])
        results.append({
            "text": text,
            "tokens_in": tokens_in,
            "tokens_out": tokens_out,
            "generation_time_s": round(elapsed / len(images), 3),
            "tokens_per_second": round(tokens_out / max(elapsed, 1e-6), 2),
            "stop_reason": "MAX_TOKENS" if tokens_out >= int(max_new_tokens) else "EOS",
            "batched": True,
            "batch_size": len(images),
        })

    del output_ids, inputs
    return results


print(f"Experimental batching defined but disabled (ENABLE_EXPERIMENTAL_BATCHING={ENABLE_EXPERIMENTAL_BATCHING}).")
print("Baseline stays at batch size 1 until single-page correctness is proven.")

## 27 — Final diagnostic summary

Everything the run produced, in one block: inventory coverage, OCR outcomes,
degeneration counts, token totals and latency percentiles.

Read it against the question the notebook set out to answer: *can this model read
these pages?* The numbers that decide it are **degenerate generations** (should
be 0), **failed pages** (should be 0) and the transcriptions themselves in §22.
The latency figures decide the separate question of whether a full run is
affordable.

In [ ]:
def final_summary(
    inventory: pd.DataFrame,
    records: Sequence[Dict[str, Any]],
    analysis: Dict[str, Any],
    elapsed_s: float,
    pdf_failures: Optional[Sequence[Dict[str, Any]]] = None,
) -> Dict[str, Any]:
    """One consolidated block of numbers for the whole session."""
    pdf_failures = list(pdf_failures or [])
    frame = records_to_dataframe(records)

    statuses: Dict[str, int] = defaultdict(int)
    for record in records:
        statuses[str(record.get("status"))] += 1

    customers_processed = len({str(record.get("customer_id")) for record in records})
    pdfs_processed = len({
        (str(record.get("customer_id")), str(record.get("physical_filename")))
        for record in records
    })

    generation_times = frame["generation_time_s"].dropna().tolist()
    total_times = frame["total_time_s"].dropna().tolist()

    summary = {
        "customers_inventoried": int(inventory["customer_id"].nunique()),
        "target_pdfs_found": int(inventory["exists"].sum()),
        "target_pdfs_missing": int((~inventory["exists"]).sum()),
        "duplicate_target_pdfs": int(inventory.loc[inventory["is_ambiguous"], "duplicate_count"].sum()
                                     - inventory["is_ambiguous"].sum()),

        "customers_ocr_processed": customers_processed,
        "pdfs_ocr_processed": pdfs_processed,
        "pdfs_failed_to_open": len(pdf_failures),
        "pages_discovered": int(len(records)),
        "pages_processed": int(statuses.get("SUCCESS", 0) + statuses.get("SUCCESS_FALLBACK", 0)
                               + statuses.get("DEGENERATE", 0)),
        "blank_pages_skipped": int(statuses.get("BLANK_PAGE", 0)),
        "successful_pages": int(statuses.get("SUCCESS", 0) + statuses.get("SUCCESS_FALLBACK", 0)),
        "failed_pages": int(statuses.get("FAILED", 0)),
        "budget_skipped_pages": int(statuses.get("SKIPPED_BUDGET", 0)),
        "degenerate_generations": int(statuses.get("DEGENERATE", 0)),
        "fallback_hd_calls": int(frame["fallback_used"].fillna(False).astype(bool).sum()),

        "total_qwen_calls": int(QWEN_CALL_COUNTER),
        "total_input_tokens": int(frame["tokens_in"].fillna(0).sum()),
        "total_output_tokens": int(frame["tokens_out"].fillna(0).sum()),

        "avg_generation_s_per_page": round(float(np.mean(generation_times)), 3) if generation_times else 0.0,
        "median_generation_s_per_page": round(float(statistics.median(generation_times)), 3) if generation_times else 0.0,
        "p95_generation_s_per_page": round(percentile(generation_times, 0.95), 3),
        "avg_total_s_per_page": round(float(np.mean(total_times)), 3) if total_times else 0.0,
        "avg_output_tokens_per_second": float(analysis.get("tokens_per_second_mean", 0.0)),

        "total_runtime_s": round(float(elapsed_s), 2),
        "status_breakdown": dict(statuses),
    }

    print("=" * 78)
    print(f"FINAL DIAGNOSTIC SUMMARY | {PIPELINE_VERSION}")
    print("=" * 78)
    print(f"Customers inventoried       : {summary['customers_inventoried']}")
    print(f"Target PDFs found           : {summary['target_pdfs_found']}")
    print(f"Target PDFs missing         : {summary['target_pdfs_missing']}")
    print(f"Duplicate target PDFs       : {summary['duplicate_target_pdfs']}")
    print()
    print(f"Customers OCR processed     : {summary['customers_ocr_processed']}")
    print(f"PDFs OCR processed          : {summary['pdfs_ocr_processed']}")
    print(f"PDFs that failed to open    : {summary['pdfs_failed_to_open']}")
    for failure in pdf_failures:
        print(f"    {failure.get('customer_id')} | {failure.get('physical_filename')} "
              f"| {failure.get('failure_type')}")
    print(f"Pages discovered            : {summary['pages_discovered']}")
    print(f"Pages processed             : {summary['pages_processed']}")
    print(f"Blank pages skipped         : {summary['blank_pages_skipped']}")
    print(f"Successful pages            : {summary['successful_pages']}")
    print(f"Failed pages                : {summary['failed_pages']}")
    print(f"Budget-skipped pages        : {summary['budget_skipped_pages']}")
    print(f"Degenerate generations      : {summary['degenerate_generations']}")
    print(f"Fallback HD calls           : {summary['fallback_hd_calls']}")
    print()
    print(f"Total Qwen calls            : {summary['total_qwen_calls']}")
    print(f"Total input tokens          : {summary['total_input_tokens']:,}")
    print(f"Total output tokens         : {summary['total_output_tokens']:,}")
    print()
    print(f"Average generation sec/page : {summary['avg_generation_s_per_page']:.3f}")
    print(f"Median generation sec/page  : {summary['median_generation_s_per_page']:.3f}")
    print(f"P95 generation sec/page     : {summary['p95_generation_s_per_page']:.3f}")
    print(f"Average total sec/page      : {summary['avg_total_s_per_page']:.3f}")
    print(f"Average output tokens/sec   : {summary['avg_output_tokens_per_second']:.2f}")
    print()
    print(f"Total runtime               : {format_duration(summary['total_runtime_s'])} "
          f"({summary['total_runtime_s']}s)")
    print("-" * 78)
    print("Status breakdown:")
    for status, count in sorted(summary["status_breakdown"].items()):
        print(f"  {status:20s} {count}")
    print("=" * 78)

    # ------------------------------------------------------------- verdict --
    print("VERDICT")
    print("-" * 78)
    if summary["degenerate_generations"] == 0 and summary["failed_pages"] == 0:
        print("  No degenerate generation, no failed page.")
        print("  The '!!!!!!!!' failure mode did not occur in this run.")
    else:
        print(f"  {summary['degenerate_generations']} degenerate and {summary['failed_pages']} "
              "failed page(s). Inspect them in section 22 before scaling up.")
    if summary["pdfs_failed_to_open"]:
        print(f"  {summary['pdfs_failed_to_open']} PDF(s) could not be opened at all "
              "(corrupt or not a PDF). Their pages were never counted.")

    if summary["successful_pages"]:
        pages_per_hour = 3600.0 / max(summary["avg_total_s_per_page"], 1e-6)
        print(f"  Sustained throughput: {pages_per_hour:.0f} pages/hour at "
              f"{summary['avg_total_s_per_page']:.1f}s/page.")
        for scale in (100, 500, 1000):
            print(f"     {scale:5d} pages -> {format_duration(summary['avg_total_s_per_page'] * scale)}")
    print("-" * 78)
    print("Next step: read the transcriptions in section 22. Timings only tell you")
    print("whether a full run is affordable, not whether the OCR is usable.")
    print("=" * 78)

    return summary


# Whichever run path was used, report its wall clock.
SESSION_ELAPSED_S = float(RUN_ELAPSED_S)
if FULL_RUN_RESULT is not None:
    SESSION_ELAPSED_S += float(FULL_RUN_RESULT.get("elapsed_s", 0.0))
    for _customer in FULL_RUN_RESULT.get("customers", []):
        for _summary in _customer.get("pdf_summaries", []):
            if _summary.get("status") == "FAILED":
                PDF_FAILURES.append({
                    "customer_id": _summary.get("customer_id"),
                    "logical_document_type": _summary.get("logical_document_type"),
                    "physical_filename": _summary.get("physical_filename"),
                    "failure_type": _summary.get("failure_type"),
                    "failure_message": _summary.get("failure_message"),
                })

# The checkpoint store is the source of truth. Section 25 runs AFTER sections 23
# and 24, so re-read it here and refresh the analysis and the outputs if it grew.
FINAL_RECORDS = load_all_checkpoints(CFG) or list(RUN_RECORDS)
if len(FINAL_RECORDS) != len(ALL_RECORDS):
    print(f"Checkpoint store changed since section 23 "
          f"({len(ALL_RECORDS)} -> {len(FINAL_RECORDS)} records): refreshing analysis and outputs.\n")
    ALL_RECORDS = FINAL_RECORDS
    PERFORMANCE = analyse_performance(FINAL_RECORDS)
    PERFORMANCE_DF = PERFORMANCE.get("frame", records_to_dataframe(FINAL_RECORDS))
    OUTPUT_PATHS = write_raw_ocr_outputs(FINAL_RECORDS, CFG)
    print()

FINAL_SUMMARY = final_summary(
    INVENTORY_DF,
    FINAL_RECORDS,
    PERFORMANCE,
    SESSION_ELAPSED_S,
    pdf_failures=PDF_FAILURES,
)

SUMMARY_PATH = write_json(CFG.OUTPUT_DIR / "kyc_raw_ocr_summary.json", {
    "generated_at": datetime.now().isoformat(timespec="seconds"),
    "pipeline_version": PIPELINE_VERSION,
    "config": {key: value for key, value in asdict(CFG).items()},
    "environment": ENV_INFO,
    "model": {
        "path": CFG.MODEL_PATH,
        "class": type(model).__name__,
        "processor_class": type(processor).__name__,
        "dtype": FIRST_PARAM_DTYPE,
        "fp8_config_source": FP8_CONFIG_SOURCE,
        "load_time_s": MODEL_LOAD_TIME_S,
        "cpu_offload_detected": CPU_OFFLOAD_DETECTED,
        "input_device": str(INPUT_DEVICE),
    },
    "benchmark": {"cold": BENCH_COLD, "warm": BENCH_WARM},
    "call_plan": CALL_PLAN,
    "summary": FINAL_SUMMARY,
    "performance": {key: value for key, value in PERFORMANCE.items() if key != "frame"},
    "errors": RUN_ERRORS,
    "pdf_failures": PDF_FAILURES,
})

print(f"\nSummary JSON: {SUMMARY_PATH}")
log(f"Session complete. Summary at {SUMMARY_PATH}")